In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')  # UTC+8 as required

# Reuse variable names across cells if re-run in the same kernel
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

def _uniq_sorted_truncated(series, limit=50, maxchars=300):
    vals = sorted(pd.Series(series).dropna().unique().tolist())
    vals_limited = vals[:limit]
    s = ','.join(map(str, vals_limited))
    if len(s) > maxchars:
        s = s[:maxchars] + '...'
    return s

def summarize_df(df, filename, name_field=None):
    row_count = int(df.shape[0])
    # compute min/max timestamp as integers when possible
    if 'timestamp' in df.columns and not df['timestamp'].dropna().empty:
        try:
            min_ts = int(df['timestamp'].min())
            max_ts = int(df['timestamp'].max())
        except Exception:
            # fallback if non-numeric
            min_ts = df['timestamp'].min()
            max_ts = df['timestamp'].max()
    else:
        min_ts = None
        max_ts = None

    cmdb_ids = _uniq_sorted_truncated(df['cmdb_id']) if 'cmdb_id' in df.columns else ''
    name_vals = _uniq_sorted_truncated(df[name_field]) if (name_field and name_field in df.columns) else ''
    sample = df.head(5).copy()
    return {
        'file': filename,
        'row_count': row_count,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'cmdb_ids_sample': cmdb_ids,
        'name_field_sample': name_vals,
        'sample_rows': sample  # keep full small sample DataFrame
    }

sum_metric = summarize_df(df_metric, 'metric.csv', name_field='kpi_name')
sum_trace = summarize_df(df_trace, 'trace.csv', name_field='trace_name')
sum_log = summarize_df(df_log, 'log.csv', name_field='log_name')
sum_error = summarize_df(df_error, 'error_logs.csv', name_field=None)

# Build a compact summary DataFrame (without the sample rows column)
summary_df = pd.DataFrame([
    {k: v for k, v in sum_metric.items() if k != 'sample_rows'},
    {k: v for k, v in sum_trace.items() if k != 'sample_rows'},
    {k: v for k, v in sum_log.items() if k != 'sample_rows'},
    {k: v for k, v in sum_error.items() if k != 'sample_rows'},
])

# Expose sample heads separately (keeps final outputs compact)
metric_head = sum_metric['sample_rows']
trace_head = sum_trace['sample_rows']
log_head = sum_log['sample_rows']
error_head = sum_error['sample_rows']

# Display compact summary first, then each file's top 5 rows as examples
summary_df, metric_head, trace_head, log_head, error_head

```
Out[1]:
```
```python
# Plain-English summary built from previously loaded DataFrames (df_metric, df_trace, df_log, df_error)
def _uniq_sorted_truncated(series, limit=50):
    vals = sorted(pd.Series(series).dropna().unique().tolist())
    return vals[:limit]

def _compact_rows(df):
    # return compact string of first 5 rows
    return df.head(5).to_string(index=False)

m_count = int(df_metric.shape[0])
t_count = int(df_trace.shape[0])
l_count = int(df_log.shape[0])
e_count = int(df_error.shape[0])

m_min, m_max = int(df_metric['timestamp'].min()), int(df_metric['timestamp'].max())
t_min, t_max = int(df_trace['timestamp'].min()), int(df_trace['timestamp'].max())
l_min, l_max = int(df_log['timestamp'].min()), int(df_log['timestamp'].max())
# error_logs may be empty
if 'timestamp' in df_error.columns and not df_error['timestamp'].dropna().empty:
    e_min, e_max = int(df_error['timestamp'].min()), int(df_error['timestamp'].max())
else:
    e_min = e_max = None

m_cmdb = _uniq_sorted_truncated(df_metric['cmdb_id'], limit=50)
t_cmdb = _uniq_sorted_truncated(df_trace['cmdb_id'], limit=50)
l_cmdb = _uniq_sorted_truncated(df_log['cmdb_id'], limit=50)
e_cmdb = _uniq_sorted_truncated(df_error['cmdb_id'], limit=50) if 'cmdb_id' in df_error.columns else []

m_names = _uniq_sorted_truncated(df_metric['kpi_name'], limit=50)
t_names = _uniq_sorted_truncated(df_trace['trace_name'], limit=50)
l_names = _uniq_sorted_truncated(df_log['log_name'], limit=50)

summary = (
    "Summary of loaded telemetry files (compact):\n\n"
    "metric.csv:\n"
    f"  - total rows: {m_count}\n"
    f"  - min_timestamp: {m_min}, max_timestamp: {m_max}\n"
    f"  - unique cmdb_id (first {min(50,len(m_cmdb))}): {m_cmdb}\n"
    f"  - unique kpi_name (first {min(50,len(m_names))}): {m_names}\n"
    "  - top 5 rows (examples):\n"
    f"{_compact_rows(metric_head)}\n\n"
    "trace.csv:\n"
    f"  - total rows: {t_count}\n"
    f"  - min_timestamp: {t_min}, max_timestamp: {t_max}\n"
    f"  - unique cmdb_id (first {min(50,len(t_cmdb))}): {t_cmdb}\n"
    f"  - unique trace_name (first {min(50,len(t_names))}): {t_names}\n"
    "  - top 5 rows (examples):\n"
    f"{_compact_rows(trace_head)}\n\n"
    "log.csv:\n"
    f"  - total rows: {l_count}\n"
    f"  - min_timestamp: {l_min}, max_timestamp: {l_max}\n"
    f"  - unique cmdb_id (first {min(50,len(l_cmdb))}): {l_cmdb}\n"
    f"  - unique log_name (first {min(50,len(l_names))}): {l_names}\n"
    "  - top 5 rows (examples):\n"
    f"{_compact_rows(log_head)}\n\n"
    "error_logs.csv:\n"
    f"  - total rows: {e_count}\n"
    f"  - min_timestamp: {e_min}, max_timestamp: {e_max}\n"
    f"  - unique cmdb_id (first {min(50,len(e_cmdb))}): {e_cmdb}\n"
    "  - top 5 rows (examples):\n"
    f"{_compact_rows(error_head) if not df_error.empty else '  (no rows)'}\n\n"
    "Notes:\n"
    "  - Timestamps are integer epoch seconds (UTC+8 context per task).\n"
    "  - Unique lists are sorted and truncated to the first 50 entries as requested.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(             file  row_count  min_timestamp  max_timestamp                                    cmdb_ids_sample                                  name_field_sample
0      metric.csv     327084   1.647873e+09   1.647877e+09  adservice,adservice-0,adservice-1,adservice-2,...  app.grpc.count,app.grpc.mrt,app.grpc.rr,app.gr...
1       trace.csv      22232   1.647875e+09   1.647877e+09  adservice-0,adservice-1,adservice-2,adservice2...  trace.from_checkoutservice-0.duration_mean,tra...
2         log.csv       1570   1.647875e+09   1.647877e+09  adservice-0,adservice-1,adservice-2,cartservic...                      log.error_count,log.row_count
3  error_logs.csv          0            NaN            NaN                                                                                                      ,     timestamp    cmdb_id        kpi_name       value
0  1647873000  adservice  app.grpc.count   79.000000
1  1647873000  adservice    app.grpc.mrt    0.002503
2  1647873000  adservice     app.grpc.rr  100.000000
3  1647873000  adservice     app.grpc.sr  100.000000
4  1647873000  adservice  app.http.count   24.000000,     timestamp      cmdb_id                           trace_name     value
0  1647874800  adservice-0  trace.from_frontend-0.duration_mean  0.000019
1  1647874800  adservice-0   trace.from_frontend-0.duration_p95  0.000021
2  1647874800  adservice-0     trace.from_frontend-0.error_rate  0.000000
3  1647874800  adservice-0      trace.from_frontend-0.row_count  7.000000
4  1647874800  adservice-0  trace.from_frontend-2.duration_mean  0.000021,     timestamp      cmdb_id         log_name  value
0  1647874800  adservice-0  log.error_count    0.0
1  1647874800  adservice-0    log.row_count   36.0
2  1647874800  adservice-1  log.error_count    0.0
3  1647874800  adservice-1    log.row_count   24.0
4  1647874800  adservice-2  log.error_count    0.0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')  # ensure UTC+8 context as required

# Reuse df_metric if already loaded in the kernel; otherwise load from file
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Group by (cmdb_id, kpi_name) and compute global statistics (no time filtering)
grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
df_metric_kpi_stats = grp.agg(
    count='count',
    mean='mean',
    min='min',
    max='max',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

# Sort by count descending and keep compact numeric formatting
df_metric_kpi_stats = df_metric_kpi_stats.sort_values(by='count', ascending=False)
df_metric_kpi_stats[['mean','min','max','p95','p5']] = df_metric_kpi_stats[['mean','min','max','p95','p5']].round(6)

# Prepare compact outputs: top 20 rows and total distinct groups
df_metric_kpi_stats_head = df_metric_kpi_stats.head(20)
groups_count = int(df_metric_kpi_stats.shape[0])

# Display the compact table and the total groups count
df_metric_kpi_stats_head, groups_count

```
Out[2]:
```
```python
# Build a plain-English summary string from the already-computed df_metric_kpi_stats and groups_count.
try:
    df_metric_kpi_stats, groups_count
except NameError:
    import pandas as pd
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    df_metric_kpi_stats = grp.agg(
        count='count',
        mean='mean',
        min='min',
        max='max',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    ).reset_index().sort_values(by='count', ascending=False)
    df_metric_kpi_stats[['mean','min','max','p95','p5']] = df_metric_kpi_stats[['mean','min','max','p95','p5']].round(6)
    groups_count = int(df_metric_kpi_stats.shape[0])

top3 = df_metric_kpi_stats.head(3)

lines = [
    f"Total distinct (cmdb_id, kpi_name) groups: {groups_count}.",
    "Computed global statistics per group: count, mean, min, max, p95, p5 (no time filtering).",
    "Top 3 groups by sample count (sorted by count desc):"
]
for _, r in top3.iterrows():
    lines.append(
        f" - {r['cmdb_id']} | {r['kpi_name']} -> "
        f"count={int(r['count'])}, mean={r['mean']}, min={r['min']}, max={r['max']}, p95={r['p95']}, p5={r['p5']}"
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count          mean           min           max           p95            p5
5373  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61   1415.110656    796.000000   1890.500000   1791.000000   1094.500000
0              adservice                                     app.grpc.count     61     75.770492     57.000000     88.000000     86.000000     66.000000
1              adservice                                       app.grpc.mrt     61      0.002871      0.002408      0.006350      0.004115      0.002464
2              adservice                                        app.grpc.rr     61    100.000000    100.000000    100.000000    100.000000    100.000000
3              adservice                                        app.grpc.sr     61    100.000000    100.000000    100.000000    100.000000    100.000000
4              adservice                                     app.http.count     61     24.000000     24.000000     24.000000     24.000000     24.000000
5              adservice                                       app.http.mrt     61      0.009250      0.008685      0.012260      0.009533      0.008954
6              adservice                                        app.http.rr     61    100.000000    100.000000    100.000000    100.000000    100.000000
7              adservice                                        app.http.sr     61    100.000000    100.000000    100.000000    100.000000    100.000000
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount     61   5162.000000   5162.000000   5162.000000   5162.000000   5162.000000
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     61   5185.000000   5185.000000   5185.000000   5185.000000   5185.000000
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     61     23.000000     23.000000     23.000000     23.000000     23.000000
11             adservice             runtime.java_lang_ClassLoading_Verbose     61      0.000000      0.000000      0.000000      0.000000      0.000000
12             adservice  runtime.java_lang_Compilation_CompilationTimeM...     61      1.000000      1.000000      1.000000      1.000000      1.000000
13             adservice  runtime.java_lang_Compilation_TotalCompilation...     61  47032.000000  47032.000000  47032.000000  47032.000000  47032.000000
14             adservice  runtime.java_lang_GarbageCollector_CollectionC...     61  40888.814208  40814.750000  40962.916667  40955.500000  40822.083333
15             adservice  runtime.java_lang_GarbageCollector_CollectionC...     61      9.000000      9.000000      9.000000      9.000000      9.000000
16             adservice  runtime.java_lang_GarbageCollector_CollectionT...     61  74321.771858  74188.916667  74454.833333  74441.000000  74202.416667
17             adservice  runtime.java_lang_GarbageCollector_CollectionT...     61    573.000000    573.000000    573.000000    573.000000    573.000000
18             adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61      1.000000      1.000000      1.000000      1.000000      1.000000, 5374)```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import datetime

# Reuse existing variables if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

try:
    df_metric_kpi_stats
except NameError:
    # recompute global thresholds if missing (must be computed on full series)
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    df_metric_kpi_stats = grp.agg(
        count='count',
        mean='mean',
        min='min',
        max='max',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()

# Ensure numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_metric_kpi_stats['p95'] = pd.to_numeric(df_metric_kpi_stats['p95'], errors='coerce')
df_metric_kpi_stats['p5'] = pd.to_numeric(df_metric_kpi_stats['p5'], errors='coerce')

# Define incident window in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2022, 3, 21, 23, 0, 0))
end_dt = tz.localize(datetime.datetime(2022, 3, 21, 23, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Filter metric rows within the incident window (inclusive)
df_incident_metric = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Join global thresholds (p95, p5) onto each metric row
df_incident_metric = df_incident_metric.merge(
    df_metric_kpi_stats[['cmdb_id', 'kpi_name', 'p95', 'p5']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark anomalies per row
df_incident_metric['high_anom'] = df_incident_metric['p95'].notna() & (df_incident_metric['value'] >= df_incident_metric['p95'])
df_incident_metric['low_anom'] = df_incident_metric['p5'].notna() & (df_incident_metric['value'] <= df_incident_metric['p5'])
df_incident_metric['anom_any'] = df_incident_metric['high_anom'] | df_incident_metric['low_anom']

# Compute per-row severities (safe for zero thresholds)
def safe_div(numer, denom):
    return numer / denom if denom != 0 else numer / 1.0

# Vectorized severity calculations with safeguards
df_incident_metric['sev_high'] = 0.0
df_incident_metric.loc[df_incident_metric['high_anom'], 'sev_high'] = (
    (df_incident_metric.loc[df_incident_metric['high_anom'], 'value'] -
     df_incident_metric.loc[df_incident_metric['high_anom'], 'p95']) /
    df_incident_metric.loc[df_incident_metric['high_anom'], 'p95'].replace(0, 1)
)

df_incident_metric['sev_low'] = 0.0
df_incident_metric.loc[df_incident_metric['low_anom'], 'sev_low'] = (
    (df_incident_metric.loc[df_incident_metric['low_anom'], 'p5'] -
     df_incident_metric.loc[df_incident_metric['low_anom'], 'value']) /
    df_incident_metric.loc[df_incident_metric['low_anom'], 'p5'].replace(0, 1)
)

# Filter to only anomaly rows for aggregation
df_anom_rows = df_incident_metric[df_incident_metric['anom_any']].copy()

if not df_anom_rows.empty:
    # Aggregate per (cmdb_id, kpi_name)
    agg = df_anom_rows.groupby(['cmdb_id', 'kpi_name']).agg(
        anomaly_count=('anom_any', 'size'),
        earliest_anom_timestamp=('timestamp', 'min'),
        high_count=('high_anom', 'sum'),
        low_count=('low_anom', 'sum'),
        max_sev_high=('sev_high', 'max'),
        max_sev_low=('sev_low', 'max')
    ).reset_index()

    # Determine anomaly_type and severity
    def determine_type(row):
        if row['high_count'] > 0 and row['low_count'] == 0:
            return 'high'
        elif row['low_count'] > 0 and row['high_count'] == 0:
            return 'low'
        elif row['high_count'] > 0 and row['low_count'] > 0:
            return 'both'
        else:
            return 'none'

    agg['anomaly_type'] = agg.apply(determine_type, axis=1)
    agg['severity'] = agg[['max_sev_high', 'max_sev_low']].max(axis=1)
    agg['severity'] = agg['severity'].clip(lower=0).round(6)

    # Keep required columns
    df_metric_anoms = agg[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_timestamp', 'anomaly_type', 'severity']].copy()
    # Sort by anomaly_count desc and keep top 20 for compact display
    df_metric_anoms = df_metric_anoms.sort_values(by='anomaly_count', ascending=False).reset_index(drop=True)
    df_metric_anoms_top = df_metric_anoms.head(20)
    num_groups_with_anoms = int(df_metric_anoms.shape[0])
else:
    # No anomalies found in the window
    df_metric_anoms = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anom_timestamp','anomaly_type','severity'])
    df_metric_anoms_top = df_metric_anoms.copy()
    num_groups_with_anoms = 0

# Final compact outputs: df_incident_metric (filtered rows), df_metric_anoms (aggregated), top20 view and count
df_metric_anoms_top, num_groups_with_anoms

```
Out[3]:
```
```python
# Plain-English summary built from the incident-window anomaly results available in the kernel.
import datetime, pytz

try:
    df_metric_anoms, df_metric_anoms_top, num_groups_with_anoms
except NameError:
    # Fallback: if variables aren't present, provide a short fallback message.
    summary = "No anomaly results found in the kernel. Run the anomaly detection cell first."
    summary
else:
    tz = pytz.timezone('Asia/Shanghai')  # UTC+8
    # derive a readable earliest timestamp from the top table if available
    if not df_metric_anoms_top.empty:
        ts = int(df_metric_anoms_top.loc[0, 'earliest_anom_timestamp'])
        earliest_dt = datetime.datetime.fromtimestamp(ts, tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_dt = 'N/A'

    # collect a few example groups from the top results
    examples = []
    for _, r in df_metric_anoms_top.head(5).iterrows():
        examples.append(f"{r['cmdb_id']} | {r['kpi_name']} (count={int(r['anomaly_count'])}, type={r['anomaly_type']}, sev={r['severity']})")

    summary_lines = [
        f"Incident window summary (2022-03-21 23:00–23:30, UTC+8):",
        f"- Total distinct (cmdb_id, kpi_name) groups with anomaly_count > 0: {int(num_groups_with_anoms)}.",
        f"- Top groups have earliest anomaly at: {earliest_dt}.",
        "- Typical top-group pattern: anomaly_count = 31 (one data point per minute across the 31-minute inclusive window).",
        "- Anomaly type for the top groups: mostly 'both' (both high and low flags occurred).",
        "- Reported severity for the top groups: 0.0 — this indicates values matched the p95/p5 thresholds (no positive relative exceedance beyond threshold).",
        "- Example top groups:",
    ] + [f"  * {e}" for e in examples]

    summary = "\n".join(summary_lines)
    summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                           kpi_name  anomaly_count  earliest_anom_timestamp anomaly_type  severity
0   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31               1647874800         both       0.0
1   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31               1647874800         both       0.0
2   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31               1647874800         both       0.0
3            adservice                                        app.grpc.rr             31               1647874800         both       0.0
4            adservice                                        app.grpc.sr             31               1647874800         both       0.0
5            adservice                                     app.http.count             31               1647874800         both       0.0
6   shippingservice2-0  mesh.destination.frontend2.shippingservice2.is...             31               1647874800         both       0.0
7            adservice                                        app.http.rr             31               1647874800         both       0.0
8            adservice                                        app.http.sr             31               1647874800         both       0.0
9            adservice    runtime.java_lang_ClassLoading_LoadedClassCount             31               1647874800         both       0.0
10           adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...             31               1647874800         both       0.0
11           adservice  runtime.java_lang_ClassLoading_UnloadedClassCount             31               1647874800         both       0.0
12           adservice             runtime.java_lang_ClassLoading_Verbose             31               1647874800         both       0.0
13           adservice  runtime.java_lang_Compilation_CompilationTimeM...             31               1647874800         both       0.0
14           adservice  runtime.java_lang_Compilation_TotalCompilation...             31               1647874800         both       0.0
15  shippingservice2-0            container.node-6.container_memory_cache             31               1647874800         both       0.0
16  shippingservice2-0          container.node-6.container_memory_failcnt             31               1647874800         both       0.0
17  shippingservice2-0  container.node-6.container_memory_failures.con...             31               1647874800         both       0.0
18  shippingservice2-0  container.node-6.container_memory_failures.con...             31               1647874800         both       0.0
19  shippingservice2-0  container.node-6.container_memory_failures.hie...             31               1647874800         both       0.0, 5343)```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse df_metric_anoms computed previously in the kernel
try:
    df_metric_anoms
except NameError:
    raise RuntimeError("df_metric_anoms not found in kernel. Run previous anomaly detection steps first.")

# Exclude KPI groups whose kpi_name contains 'mesh.'
df_anoms_no_mesh = df_metric_anoms[~df_metric_anoms['kpi_name'].str.contains('mesh.', na=False)].copy()

# Build the set of provided root-cause candidate cmdb_id values
nodes = [f"node-{i}" for i in range(1,7)]
pods = [
    "frontend-0","frontend-1","frontend-2","frontend2-0",
    "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
    "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
    "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
    "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
    "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
    "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
]
services = [
    "frontend","shippingservice","checkoutservice","currencyservice",
    "adservice","emailservice","cartservice","productcatalogservice",
    "recommendationservice","paymentservice"
]
candidates_set = set(nodes + pods + services)

# Filter to only candidate cmdb_id values
df_anoms_candidates = df_anoms_no_mesh[df_anoms_no_mesh['cmdb_id'].isin(candidates_set)].copy()

# If there are no rows after filtering, prepare empty outputs
if df_anoms_candidates.empty:
    df_cmdb_agg = pd.DataFrame(columns=[
        'cmdb_id','distinct_kpi_anom_count','total_anomaly_count','earliest_anom_timestamp','max_severity'
    ])
    df_cmdb_agg_top = df_cmdb_agg.copy()
    num_components_with_anoms = 0
else:
    # Aggregate by cmdb_id
    agg = df_anoms_candidates.groupby('cmdb_id').agg(
        distinct_kpi_anom_count = ('kpi_name', 'nunique'),
        total_anomaly_count = ('anomaly_count', 'sum'),
        earliest_anom_timestamp = ('earliest_anom_timestamp', 'min'),
        max_severity = ('severity', 'max')
    ).reset_index()

    # Sort by distinct_kpi_anom_count desc then total_anomaly_count desc
    agg = agg.sort_values(by=['distinct_kpi_anom_count','total_anomaly_count'], ascending=[False, False]).reset_index(drop=True)

    # Round severity for compactness
    agg['max_severity'] = agg['max_severity'].round(6)

    # Keep top 20 for display
    df_cmdb_agg = agg
    df_cmdb_agg_top = df_cmdb_agg.head(20)

    # Count components with distinct_kpi_anom_count > 0
    num_components_with_anoms = int((df_cmdb_agg['distinct_kpi_anom_count'] > 0).sum())

# Compact outputs (top 20 rows and the count)
df_cmdb_agg_top, num_components_with_anoms

```
Out[4]:
```
```python
# Build a plain-English summary from df_cmdb_agg_top and num_components_with_anoms already in the kernel.
import datetime, pytz

try:
    df_cmdb_agg_top, num_components_with_anoms
except NameError:
    summary = "Required results (df_cmdb_agg_top or num_components_with_anoms) not found in kernel."
else:
    tz = pytz.timezone('Asia/Shanghai')  # UTC+8
    # Earliest timestamp shown in the table (they are same for top rows)
    if not df_cmdb_agg_top.empty:
        ts = int(df_cmdb_agg_top.iloc[0]['earliest_anom_timestamp'])
        earliest_dt = datetime.datetime.fromtimestamp(ts, tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_dt = 'N/A'

    # Prepare top entries (show up to 10 compactly)
    top_n = min(10, len(df_cmdb_agg_top))
    top_lines = []
    for i in range(top_n):
        row = df_cmdb_agg_top.iloc[i]
        top_lines.append(
            f"{i+1}. {row['cmdb_id']} — distinct_kpi_anom_count={int(row['distinct_kpi_anom_count'])}, "
            f"total_anomaly_count={int(row['total_anomaly_count'])}, max_severity={float(row['max_severity']):.6f}"
        )

    summary_lines = [
        "Summary of filtered anomaly aggregation (mesh.* excluded; only candidate components kept):",
        f"- Total distinct components with >=1 anomalous KPI: {int(num_components_with_anoms)}.",
        f"- Earliest anomaly observed (top groups): {earliest_dt}.",
        "- Top components by number of distinct anomalous KPIs (examples):"
    ] + top_lines + [
        "",
        "Notes:",
        "- distinct_kpi_anom_count = number of distinct KPI names with anomalies for that component.",
        "- total_anomaly_count = sum of anomaly rows across those KPIs (higher means more anomalous datapoints).",
        "- max_severity = maximum per-group severity observed across its KPIs (rounded).",
    ]

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  distinct_kpi_anom_count  total_anomaly_count  earliest_anom_timestamp  max_severity
0                  adservice                      340                 9700               1647874800      0.543135
1          shippingservice-2                       64                 1747               1647874800      1.639518
2          shippingservice-0                       64                 1742               1647874800      8.430720
3          shippingservice-1                       64                 1739               1647874800      0.268054
4             emailservice-0                       64                 1735               1647874800     20.932420
5                 frontend-2                       64                 1722               1647874800      3.223290
6    recommendationservice-2                       64                 1721               1647874800      8.386619
7         shippingservice2-0                       64                 1720               1647874800      1.000000
8             emailservice-1                       64                 1690               1647874800      0.500000
9    recommendationservice-1                       64                 1686               1647874800      8.369479
10            emailservice-2                       64                 1683               1647874800     21.506061
11  productcatalogservice2-0                       64                 1681               1647874800      0.363636
12               adservice-0                       64                 1665               1647874800      0.250000
13  recommendationservice2-0                       64                 1659               1647874800      3.381145
14          paymentservice-2                       64                 1655               1647874800      1.000000
15              adservice2-0                       64                 1645               1647874800      8.839344
16               adservice-1                       64                 1641               1647874800      7.891921
17         checkoutservice-0                       64                 1622               1647874800     11.132236
18         currencyservice-2                       64                 1600               1647874800      0.333333
19               frontend2-0                       64                 1588               1647874800      0.250000, 56)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')  # UTC+8 context as required

# Reuse df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure numeric values
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Group by (cmdb_id, trace_name) and compute global statistics (no time filtering)
grp = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
df_trace_kpi_stats = grp.agg(
    count='count',
    mean='mean',
    min='min',
    max='max',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

# Sort by count descending and round numeric columns for compactness
df_trace_kpi_stats = df_trace_kpi_stats.sort_values(by='count', ascending=False)
df_trace_kpi_stats[['mean','min','max','p95','p5']] = df_trace_kpi_stats[['mean','min','max','p95','p5']].round(6)

# Prepare compact outputs
df_trace_kpi_stats_head = df_trace_kpi_stats.head(20)
groups_count = int(df_trace_kpi_stats.shape[0])

# Return the compact table and the total groups count
df_trace_kpi_stats_head, groups_count

```
Out[5]:
```
```python
# Plain-English summary of the trace group statistics computed earlier.
try:
    df_trace_kpi_stats_head, groups_count
except NameError:
    # fallback: try to use full df_trace_kpi_stats if head not present
    try:
        df_trace_kpi_stats
        df_trace_kpi_stats_head = df_trace_kpi_stats.head(20)
        groups_count = int(df_trace_kpi_stats.shape[0])
    except NameError:
        summary = "Trace statistics not found in the kernel. Re-run the trace stats computation."
else:
    top3 = df_trace_kpi_stats_head.head(3)
    examples = []
    for _, r in top3.iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['trace_name']} -> count={int(r['count'])}, mean={r['mean']}, p95={r['p95']}, p5={r['p5']}"
        )
    summary_lines = [
        f"Trace statistics (global, no time filter):",
        f"- Total distinct (cmdb_id, trace_name) groups: {int(groups_count)}.",
        "- Each group includes: count, mean, min, max, p95, p5.",
        "- Top groups by count (examples):"
    ] + [f"  * {e}" for e in examples]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                      trace_name  count       mean        min        max        p95         p5
984  recommendationservice2-0  trace.to_productcatalogservice-2.duration_mean     30   0.000031   0.000028   0.000043   0.000035   0.000029
985  recommendationservice2-0   trace.to_productcatalogservice-2.duration_p95     30   0.000038   0.000031   0.000057   0.000049   0.000032
986  recommendationservice2-0     trace.to_productcatalogservice-2.error_rate     30   0.000000   0.000000   0.000000   0.000000   0.000000
987  recommendationservice2-0      trace.to_productcatalogservice-2.row_count     30  18.200000  14.000000  22.000000  21.550000  15.000000
976  recommendationservice2-0  trace.to_productcatalogservice-0.duration_mean     30   0.000029   0.000028   0.000032   0.000032   0.000028
977  recommendationservice2-0   trace.to_productcatalogservice-0.duration_p95     30   0.000035   0.000030   0.000043   0.000040   0.000030
978  recommendationservice2-0     trace.to_productcatalogservice-0.error_rate     30   0.000000   0.000000   0.000000   0.000000   0.000000
979  recommendationservice2-0      trace.to_productcatalogservice-0.row_count     30  18.233333  14.000000  23.000000  20.550000  15.450000
980  recommendationservice2-0  trace.to_productcatalogservice-1.duration_mean     30   0.000031   0.000028   0.000046   0.000043   0.000028
981  recommendationservice2-0   trace.to_productcatalogservice-1.duration_p95     30   0.000040   0.000031   0.000093   0.000061   0.000031
982  recommendationservice2-0     trace.to_productcatalogservice-1.error_rate     30   0.000000   0.000000   0.000000   0.000000   0.000000
983  recommendationservice2-0      trace.to_productcatalogservice-1.row_count     30  18.200000  13.000000  23.000000  21.000000  15.000000
968  recommendationservice2-0            trace.from_frontend2-0.duration_mean     30   0.002721   0.002639   0.002867   0.002786   0.002645
969  recommendationservice2-0             trace.from_frontend2-0.duration_p95     30   0.003253   0.003024   0.003516   0.003476   0.003067
970  recommendationservice2-0               trace.from_frontend2-0.error_rate     30   0.000000   0.000000   0.000000   0.000000   0.000000
971  recommendationservice2-0                trace.from_frontend2-0.row_count     30  54.633333  41.000000  68.000000  63.100000  45.900000
972  recommendationservice2-0                        trace.self.duration_mean     30   0.002340   0.002259   0.002488   0.002400   0.002266
973  recommendationservice2-0                         trace.self.duration_p95     30   0.002848   0.002638   0.003121   0.003075   0.002685
974  recommendationservice2-0                           trace.self.error_rate     30   0.000000   0.000000   0.000000   0.000000   0.000000
975  recommendationservice2-0                            trace.self.row_count     30  54.633333  41.000000  68.000000  63.100000  45.900000, 1056)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import datetime

tz = pytz.timezone('Asia/Shanghai')  # UTC+8

# Reuse existing variables if present; otherwise load/compute as needed
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# Ensure trace global thresholds exist
try:
    df_trace_kpi_stats
except NameError:
    df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
    grp = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
    df_trace_kpi_stats = grp.agg(
        count='count',
        mean='mean',
        min='min',
        max='max',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()

# Ensure numeric types
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_trace_kpi_stats['p95'] = pd.to_numeric(df_trace_kpi_stats['p95'], errors='coerce')
df_trace_kpi_stats['p5'] = pd.to_numeric(df_trace_kpi_stats['p5'], errors='coerce')

# Incident window in UTC+8: 2022-03-21 23:00 to 23:30 inclusive
start_dt = tz.localize(datetime.datetime(2022, 3, 21, 23, 0, 0))
end_dt = tz.localize(datetime.datetime(2022, 3, 21, 23, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Filter trace rows within the incident window (inclusive)
df_trace_incident = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Join global thresholds onto each trace row
df_trace_incident = df_trace_incident.merge(
    df_trace_kpi_stats[['cmdb_id', 'trace_name', 'p95', 'p5']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Mark anomalies per row
df_trace_incident['high_anom'] = df_trace_incident['p95'].notna() & (df_trace_incident['value'] >= df_trace_incident['p95'])
df_trace_incident['low_anom'] = df_trace_incident['p5'].notna() & (df_trace_incident['value'] <= df_trace_incident['p5'])
df_trace_incident['anom_any'] = df_trace_incident['high_anom'] | df_trace_incident['low_anom']

# Compute per-row severities with safe denominators
df_trace_incident['sev_high'] = 0.0
mask_high = df_trace_incident['high_anom']
df_trace_incident.loc[mask_high, 'sev_high'] = (
    (df_trace_incident.loc[mask_high, 'value'] - df_trace_incident.loc[mask_high, 'p95']) /
    df_trace_incident.loc[mask_high, 'p95'].replace(0, 1)
)

df_trace_incident['sev_low'] = 0.0
mask_low = df_trace_incident['low_anom']
df_trace_incident.loc[mask_low, 'sev_low'] = (
    (df_trace_incident.loc[mask_low, 'p5'] - df_trace_incident.loc[mask_low, 'value']) /
    df_trace_incident.loc[mask_low, 'p5'].replace(0, 1)
)

# Keep incident-window filtered traces in kernel variable
df_trace_incident_window = df_trace_incident  # stored variable

# Filter to only anomaly rows for aggregation
df_trace_anom_rows = df_trace_incident_window[df_trace_incident_window['anom_any']].copy()

if not df_trace_anom_rows.empty:
    agg = df_trace_anom_rows.groupby(['cmdb_id', 'trace_name']).agg(
        anomaly_count=('anom_any', 'size'),
        earliest_anom_timestamp=('timestamp', 'min'),
        high_count=('high_anom', 'sum'),
        low_count=('low_anom', 'sum'),
        max_sev_high=('sev_high', 'max'),
        max_sev_low=('sev_low', 'max')
    ).reset_index()

    # Determine anomaly_type and severity
    def _anomaly_type(row):
        if row['high_count'] > 0 and row['low_count'] == 0:
            return 'high'
        if row['low_count'] > 0 and row['high_count'] == 0:
            return 'low'
        if row['high_count'] > 0 and row['low_count'] > 0:
            return 'both'
        return 'none'

    agg['anomaly_type'] = agg.apply(_anomaly_type, axis=1)
    agg['severity'] = agg[['max_sev_high', 'max_sev_low']].max(axis=1).clip(lower=0).round(6)

    df_trace_anoms = agg[['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anom_timestamp', 'anomaly_type', 'severity']].copy()
    df_trace_anoms = df_trace_anoms.sort_values(by='anomaly_count', ascending=False).reset_index(drop=True)
    df_trace_anoms_top = df_trace_anoms.head(20)
    num_trace_groups_with_anoms = int(df_trace_anoms.shape[0])
else:
    # No anomalies found
    df_trace_anoms = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anom_timestamp','anomaly_type','severity'])
    df_trace_anoms_top = df_trace_anoms.copy()
    num_trace_groups_with_anoms = 0

# Final compact outputs (top 20 aggregated trace anomaly groups and count)
df_trace_anoms_top, num_trace_groups_with_anoms

```
Out[6]:
```
```python
# Plain-English summary of trace anomalies (from df_trace_anoms_top and num_trace_groups_with_anoms)
import datetime, pytz

try:
    df_trace_anoms_top, num_trace_groups_with_anoms
except NameError:
    summary = "Trace anomaly results not found in kernel. Re-run the trace anomaly detection step."
else:
    tz = pytz.timezone('Asia/Shanghai')  # UTC+8
    # derive earliest timestamp among top rows (they are all the same in the sample)
    if not df_trace_anoms_top.empty:
        earliest_ts = int(df_trace_anoms_top['earliest_anom_timestamp'].min())
        earliest_dt = datetime.datetime.fromtimestamp(earliest_ts, tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_dt = 'N/A'

    # Prepare up to 5 example top groups
    examples = []
    for _, r in df_trace_anoms_top.head(5).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['trace_name']} — count={int(r['anomaly_count'])}, type={r['anomaly_type']}, sev={float(r['severity']):.6f}"
        )

    summary_lines = [
        "Trace anomaly summary for incident window (2022-03-21 23:00–23:30, UTC+8):",
        f"- Total distinct (cmdb_id, trace_name) groups with anomalies: {int(num_trace_groups_with_anoms)}.",
        f"- Earliest anomaly seen (sample top groups): {earliest_dt}.",
        f"- Top groups in the sample all have anomaly_count = 30 and anomaly_type = 'both' (both high and low flags).",
        "- Severity values are generally 0.0 for many error_rate traces (indicating values match threshold),",
        "  but some duration-related traces show non-zero severity (example below).",
        "- Example top trace groups:",
    ] + [f"  * {e}" for e in examples] + [
        "",
        "Notes:",
        "- anomaly_count = number of anomalous rows in the window for that (cmdb_id, trace_name).",
        "- anomaly_type = 'high'/'low'/'both' depending on which side(s) triggered.",
        "- severity = maximum relative deviation from the global p95/p5 threshold for that group."
    ]

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                   trace_name  anomaly_count  earliest_anom_timestamp anomaly_type  severity
0     adservice-0             trace.from_frontend-2.error_rate             30               1647874800         both  0.000000
1     adservice-0             trace.from_frontend-0.error_rate             30               1647874800         both  0.000000
2     adservice-1             trace.from_frontend-2.error_rate             30               1647874800         both  0.000000
3      frontend-0            trace.to_cartservice-1.error_rate             30               1647874800         both  0.000000
4      frontend-0        trace.to_currencyservice-1.error_rate             30               1647874800         both  0.000000
5      frontend-0        trace.to_currencyservice-2.error_rate             30               1647874800         both  0.000000
6      frontend-0        trace.to_currencyservice-0.error_rate             30               1647874800         both  0.000000
7      frontend-2            trace.to_cartservice-0.error_rate             30               1647874800         both  0.000000
8      frontend-0              trace.to_adservice-1.error_rate             30               1647874800         both  0.000000
9      frontend-0                   trace.from_root.error_rate             30               1647874800         both  0.000000
10     frontend-0                        trace.self.error_rate             30               1647874800         both  0.000000
11     frontend-0              trace.to_adservice-0.error_rate             30               1647874800         both  0.000000
12     frontend-0            trace.to_cartservice-0.error_rate             30               1647874800         both  0.000000
13  cartservice-1                      trace.self.duration_p95             30               1647874800         both  0.000300
14  cartservice-1                     trace.self.duration_mean             30               1647874800         both  1.791667
15  cartservice-2             trace.from_frontend-0.error_rate             30               1647874800         both  0.000000
16  cartservice-0             trace.from_frontend-2.error_rate             30               1647874800         both  0.000000
17  cartservice-1                        trace.self.error_rate             30               1647874800         both  0.000000
18  cartservice-2             trace.from_frontend-2.error_rate             30               1647874800         both  0.000000
19     frontend-0  trace.to_productcatalogservice-2.error_rate             30               1647874800         both  0.000000, 1056)```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse variables from the kernel
try:
    df_cmdb_agg
except NameError:
    raise RuntimeError("df_cmdb_agg not found in kernel. Please run previous metric aggregation steps.")

try:
    df_trace_anoms
except NameError:
    raise RuntimeError("df_trace_anoms not found in kernel. Please run previous trace anomaly detection steps.")

# Take top 20 components by distinct_kpi_anom_count (ensure sorted)
df_cmdb_top20 = df_cmdb_agg.sort_values(
    by=['distinct_kpi_anom_count', 'total_anomaly_count'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Prepare trace anomaly flags for aggregation
df_trace_anoms['is_to'] = df_trace_anoms['trace_name'].str.contains('trace.to_', na=False)
df_trace_anoms['is_from'] = df_trace_anoms['trace_name'].str.contains('trace.from_', na=False)
df_trace_anoms['is_self'] = df_trace_anoms['trace_name'].str.contains('trace.self', na=False)

# Aggregate trace anomaly metrics per cmdb_id
trace_agg = df_trace_anoms.groupby('cmdb_id').agg(
    trace_to_anom_count = ('anomaly_count', lambda s: s[df_trace_anoms.loc[s.index, 'is_to']].sum() if not s.empty else 0),
    trace_from_anom_count = ('anomaly_count', lambda s: s[df_trace_anoms.loc[s.index, 'is_from']].sum() if not s.empty else 0),
    trace_self_anom_count = ('anomaly_count', lambda s: s[df_trace_anoms.loc[s.index, 'is_self']].sum() if not s.empty else 0),
    trace_earliest_anom_timestamp = ('earliest_anom_timestamp', 'min'),
    max_trace_severity = ('severity', 'max')
).reset_index()

# The above lambdas reference df_trace_anoms by index; ensure correct alignment by rebuilding with boolean masks more robustly
# Safer explicit aggregation:
trace_agg_safe = df_trace_anoms.groupby('cmdb_id').apply(
    lambda g: pd.Series({
        'trace_to_anom_count': int(g.loc[g['is_to'], 'anomaly_count'].sum()) if not g.loc[g['is_to']].empty else 0,
        'trace_from_anom_count': int(g.loc[g['is_from'], 'anomaly_count'].sum()) if not g.loc[g['is_from']].empty else 0,
        'trace_self_anom_count': int(g.loc[g['is_self'], 'anomaly_count'].sum()) if not g.loc[g['is_self']].empty else 0,
        'trace_earliest_anom_timestamp': int(g['earliest_anom_timestamp'].min()) if not g['earliest_anom_timestamp'].isnull().all() else pd.NA,
        'max_trace_severity': float(g['severity'].max()) if not g['severity'].isnull().all() else 0.0
    })
).reset_index()

# Merge the trace aggregates onto the top20 metric components
df_component_trace_summary = df_cmdb_top20.merge(
    trace_agg_safe,
    on='cmdb_id',
    how='left'
).fillna({
    'trace_to_anom_count': 0,
    'trace_from_anom_count': 0,
    'trace_self_anom_count': 0,
    'trace_earliest_anom_timestamp': pd.NA,
    'max_trace_severity': 0.0
})

# Ensure integer types for counts and severity rounding
df_component_trace_summary['trace_to_anom_count'] = df_component_trace_summary['trace_to_anom_count'].astype(int)
df_component_trace_summary['trace_from_anom_count'] = df_component_trace_summary['trace_from_anom_count'].astype(int)
df_component_trace_summary['trace_self_anom_count'] = df_component_trace_summary['trace_self_anom_count'].astype(int)
df_component_trace_summary['max_trace_severity'] = df_component_trace_summary['max_trace_severity'].astype(float).round(6)

# Select and sort required columns
df_component_trace_summary = df_component_trace_summary[
    ['cmdb_id', 'distinct_kpi_anom_count', 'total_anomaly_count',
     'trace_to_anom_count', 'trace_from_anom_count', 'trace_self_anom_count',
     'trace_earliest_anom_timestamp', 'max_trace_severity']
].sort_values(by=['distinct_kpi_anom_count', 'max_trace_severity'], ascending=[False, False]).reset_index(drop=True)

# Limit to max 20 rows (already top20)
df_component_trace_summary = df_component_trace_summary.head(20)

# Count components that have trace_to_anom_count == 0 and (trace_from_anom_count > 0 or trace_self_anom_count > 0)
downstream_candidates_count = int(
    ((df_component_trace_summary['trace_to_anom_count'] == 0) &
     ((df_component_trace_summary['trace_from_anom_count'] > 0) | (df_component_trace_summary['trace_self_anom_count'] > 0)))
    .sum()
)

# Final compact outputs
df_component_trace_summary, downstream_candidates_count

```
Out[7]:
```
```python
# Plain-English summary of the component vs. trace-anomaly cross-check (based on df_component_trace_summary and downstream_candidates_count)
try:
    df_component_trace_summary, downstream_candidates_count
except NameError:
    summary = "Required results not found in kernel. Re-run the previous aggregation steps."
else:
    # Identify downstream-candidate components (trace_to == 0 and (trace_from >0 or trace_self>0))
    mask_downstream = (df_component_trace_summary['trace_to_anom_count'] == 0) & \
                      ((df_component_trace_summary['trace_from_anom_count'] > 0) | (df_component_trace_summary['trace_self_anom_count'] > 0))
    downstream_list = df_component_trace_summary.loc[mask_downstream, 'cmdb_id'].tolist()
    # Pick a few notable examples from the top table
    examples = []
    # adservice is the top metric-anomalous component but has no trace anomalies in this window
    row_ad = df_component_trace_summary.loc[df_component_trace_summary['cmdb_id']=='adservice']
    if not row_ad.empty:
        r = row_ad.iloc[0]
        examples.append(f"adservice: {int(r['distinct_kpi_anom_count'])} distinct KPI anomalies, {int(r['total_anomaly_count'])} total metric anomalies, but zero trace anomalies (no trace evidence).")
    # pick recommendationservice-1 as high trace severity example
    if 'recommendationservice-1' in df_component_trace_summary['cmdb_id'].values:
        r = df_component_trace_summary[df_component_trace_summary['cmdb_id']=='recommendationservice-1'].iloc[0]
        examples.append(f"recommendationservice-1: trace_to={int(r['trace_to_anom_count'])}, trace_from={int(r['trace_from_anom_count'])}, trace_self={int(r['trace_self_anom_count'])}, max_trace_severity={r['max_trace_severity']:.6f}.")
    # pick frontend-2 as high trace_to example
    if 'frontend-2' in df_component_trace_summary['cmdb_id'].values:
        r = df_component_trace_summary[df_component_trace_summary['cmdb_id']=='frontend-2'].iloc[0]
        examples.append(f"frontend-2: trace_to={int(r['trace_to_anom_count'])}, trace_from={int(r['trace_from_anom_count'])}, max_trace_severity={r['max_trace_severity']:.6f}.")
    # Build summary
    summary_lines = [
        "Summary of top-20 metric-anomalous components and their trace-anomaly patterns (incident window):",
        f"- Examined 20 top components by distinct_kpi_anom_count.",
        f"- Components meeting downstream-candidate criteria (trace_to == 0 AND (trace_from > 0 OR trace_self > 0)): {int(downstream_candidates_count)}.",
        f"- Those components (up to 20 shown): {downstream_list[:20]}",
        "",
        "- Notable observations / examples:",
    ] + [f"  * {e}" for e in examples] + [
        "",
        "Interpretation hints:",
        "- A component with many metric anomalies but zero trace anomalies (e.g., adservice above) suggests metric-side issues not reflected in traces — investigate metrics/logs/hosts.",
        "- Components with trace_from/trace_self anomalies but no trace_to anomalies (the downstream candidates listed) may be impacted downstream (they receive anomalous requests) rather than being the upstream origin.",
        "- recommendationservice-1 and frontend-2 show substantial trace_to/anom activity and higher max_trace_severity — these are strong candidates for upstream investigation.",
    ]
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  distinct_kpi_anom_count  total_anomaly_count  trace_to_anom_count  trace_from_anom_count  trace_self_anom_count  trace_earliest_anom_timestamp  max_trace_severity
0                  adservice                      340                 9700                    0                      0                      0                            NaN            0.000000
1    recommendationservice-1                       64                 1686                  156                     94                     48                   1.647875e+09            5.463415
2          shippingservice-2                       64                 1747                    0                    186                      0                   1.647875e+09            2.406593
3                 frontend-2                       64                 1722                 1010                     43                     42                   1.647875e+09            2.406593
4          currencyservice-2                       64                 1600                    0                    215                      0                   1.647875e+09            2.073892
5                adservice-1                       64                 1641                    0                    107                      0                   1.647875e+09            1.571429
6          shippingservice-1                       64                 1739                    0                    184                      0                   1.647875e+09            1.450549
7                adservice-0                       64                 1665                    0                    115                      0                   1.647875e+09            1.423077
8          checkoutservice-0                       64                 1622                  591                     78                     27                   1.647875e+09            0.904762
9          shippingservice-0                       64                 1742                    0                    203                      0                   1.647875e+09            0.815951
10        shippingservice2-0                       64                 1720                    0                     90                      0                   1.647875e+09            0.706522
11  recommendationservice2-0                       64                 1659                  151                     42                     43                   1.647875e+09            0.524590
12  productcatalogservice2-0                       64                 1681                    0                     86                      0                   1.647875e+09            0.500000
13               frontend2-0                       64                 1588                  329                     42                     43                   1.647875e+09            0.500000
14          paymentservice-2                       64                 1655                    0                     56                      0                   1.647875e+09            0.303502
15              adservice2-0                       64                 1645                    0                     47                      0                   1.647875e+09            0.244444
16   recommendationservice-2                       64                 1721                  153                     94                     42                   1.647875e+09            0.243902
17            emailservice-0                       64                 1735                    0                     52                      0                   1.647875e+09            0.089231
18            emailservice-2                       64                 1683                    0                     58                      0                   1.647875e+09            0.020478
19            emailservice-1                       64                 1690                    0                     54                      0                   1.647875e+09            0.018116, 13)```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import datetime
import numpy as np

# Reuse existing df_log if present
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure numeric values
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# 1) Compute global thresholds on full log.csv grouped by (cmdb_id, log_name)
grp = df_log.groupby(['cmdb_id', 'log_name'])['value']
df_log_kpi_stats = grp.agg(
    count='count',
    mean='mean',
    min='min',
    max='max',
    p95=lambda x: x.quantile(0.95),
    p5=lambda x: x.quantile(0.05)
).reset_index()

# Round numeric columns for compactness
df_log_kpi_stats[['mean','min','max','p95','p5']] = df_log_kpi_stats[['mean','min','max','p95','p5']].round(6)

# 2) Define incident window (UTC+8) and filter log rows in that window (inclusive)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2022,3,21,23,0,0))
end_dt = tz.localize(datetime.datetime(2022,3,21,23,30,0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

df_incident_log = df_log[(df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)].copy()

# 3) Join per-group thresholds (p95, p5) onto incident-window log rows
df_incident_log = df_incident_log.merge(
    df_log_kpi_stats[['cmdb_id','log_name','p95','p5']],
    on=['cmdb_id','log_name'],
    how='left'
)

# 4) Mark anomalies per row: high if value >= p95, low if value <= p5
df_incident_log['high_anom'] = df_incident_log['p95'].notna() & (df_incident_log['value'] >= df_incident_log['p95'])
df_incident_log['low_anom'] = df_incident_log['p5'].notna() & (df_incident_log['value'] <= df_incident_log['p5'])
df_incident_log['anom_any'] = df_incident_log['high_anom'] | df_incident_log['low_anom']

# 5) Compute per-row severities with safe denominators
# sev_high = (value - p95)/p95  (use denom=1 if p95==0)
df_incident_log['p95_safe'] = df_incident_log['p95'].replace({0:1})
df_incident_log['p5_safe'] = df_incident_log['p5'].replace({0:1})

df_incident_log['sev_high'] = 0.0
mask_h = df_incident_log['high_anom']
df_incident_log.loc[mask_h, 'sev_high'] = (
    (df_incident_log.loc[mask_h, 'value'] - df_incident_log.loc[mask_h, 'p95']) /
    df_incident_log.loc[mask_h, 'p95_safe']
)

df_incident_log['sev_low'] = 0.0
mask_l = df_incident_log['low_anom']
df_incident_log.loc[mask_l, 'sev_low'] = (
    (df_incident_log.loc[mask_l, 'p5'] - df_incident_log.loc[mask_l, 'value']) /
    df_incident_log.loc[mask_l, 'p5_safe']
)

# per-row severity = max(sev_high, sev_low)
df_incident_log['row_severity'] = df_incident_log[['sev_high','sev_low']].max(axis=1).clip(lower=0)

# 6) Determine top 20 components by distinct_kpi_anom_count from df_cmdb_agg or fallback
try:
    df_cmdb_agg
    source_df = df_cmdb_agg
except NameError:
    try:
        df_cmdb_agg_top
        source_df = df_cmdb_agg_top
    except NameError:
        # If neither exists, derive a fallback: use top 20 cmdb_id by metric anomaly count if available
        if 'df_metric_anoms' in globals():
            source_df = df_metric_anoms.groupby('cmdb_id').agg(distinct_kpi_anom_count=('kpi_name','nunique'),
                                                               total_anomaly_count=('anomaly_count','sum')).reset_index()
        else:
            # final fallback: use unique cmdb_ids from logs
            source_df = pd.DataFrame({'cmdb_id': sorted(df_log['cmdb_id'].dropna().unique())}) 

# Ensure we have distinct_kpi_anom_count in source_df (if not, set to 0)
if 'distinct_kpi_anom_count' not in source_df.columns:
    source_df = source_df.copy()
    source_df['distinct_kpi_anom_count'] = 0
if 'total_anomaly_count' not in source_df.columns:
    source_df['total_anomaly_count'] = 0

top20_components = source_df.sort_values(by='distinct_kpi_anom_count', ascending=False).head(20)['cmdb_id'].tolist()

# 7) Restrict incident logs to these top components for analysis
df_incident_log_top = df_incident_log[df_incident_log['cmdb_id'].isin(top20_components)].copy()

# 8) Aggregate per-component over anomalous rows only
df_anom_logs = df_incident_log_top[df_incident_log_top['anom_any']].copy()

if not df_anom_logs.empty:
    agg = df_anom_logs.groupby('cmdb_id').agg(
        anomalous_log_count = ('anom_any', 'size'),
        distinct_log_names_with_anom = ('log_name', 'nunique'),
        earliest_anom_timestamp = ('timestamp', 'min'),
        max_log_severity = ('row_severity', 'max')
    ).reset_index()
    agg['max_log_severity'] = agg['max_log_severity'].round(6)
else:
    agg = pd.DataFrame(columns=['cmdb_id','anomalous_log_count','distinct_log_names_with_anom','earliest_anom_timestamp','max_log_severity'])

# 9) Ensure all top components appear in the result (with zeros if no anomalies)
df_top_components_df = pd.DataFrame({'cmdb_id': top20_components})
df_log_anoms_by_component = df_top_components_df.merge(agg, on='cmdb_id', how='left').fillna({
    'anomalous_log_count': 0,
    'distinct_log_names_with_anom': 0,
    'earliest_anom_timestamp': pd.NA,
    'max_log_severity': 0.0
})
# Cast numeric types
df_log_anoms_by_component['anomalous_log_count'] = df_log_anoms_by_component['anomalous_log_count'].astype(int)
df_log_anoms_by_component['distinct_log_names_with_anom'] = df_log_anoms_by_component['distinct_log_names_with_anom'].astype(int)
df_log_anoms_by_component['max_log_severity'] = df_log_anoms_by_component['max_log_severity'].astype(float).round(6)

# 10) Prepare compact sorted output (limit 20 rows)
df_log_anoms_by_component = df_log_anoms_by_component.sort_values(by='anomalous_log_count', ascending=False).reset_index(drop=True)
df_log_anoms_by_component_top = df_log_anoms_by_component.head(20)

# 11) List of top components that have anomalous_log_count > 0 and the total count
anomalous_components_list = df_log_anoms_by_component_top.loc[df_log_anoms_by_component_top['anomalous_log_count']>0, 'cmdb_id'].tolist()
num_anomalous_components = len(anomalous_components_list)

# Final compact outputs: df_log_kpi_stats (stored), df_incident_log (stored), df_log_anoms_by_component (stored)
# Return compact table, list, and count
df_log_anoms_by_component_top, anomalous_components_list, num_anomalous_components

```
Out[8]:
```
```python
# Build a plain-English summary string from the available kernel results and display it.
import datetime, pytz

try:
    df_log_anoms_by_component_top, anomalous_components_list, num_anomalous_components
except NameError:
    summary = "Log-anomaly results not found in kernel. Re-run the log anomaly detection step."
else:
    tz = pytz.timezone('Asia/Shanghai')
    # derive readable earliest timestamp from top rows if available
    earliest_ts_vals = df_log_anoms_by_component_top['earliest_anom_timestamp'].dropna().unique().tolist()
    if earliest_ts_vals:
        earliest_dt = datetime.datetime.fromtimestamp(int(earliest_ts_vals[0]), tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_dt = 'N/A'

    # Prepare top 5 components by anomalous_log_count for examples
    examples = []
    top_rows = df_log_anoms_by_component_top.head(5).itertuples()
    for r in top_rows:
        examples.append(
            f"{r.cmdb_id}: anomalous_log_count={int(r.anomalous_log_count)}, "
            f"distinct_log_names_with_anom={int(r.distinct_log_names_with_anom)}, "
            f"max_log_severity={float(r.max_log_severity):.6f}"
        )

    summary_lines = [
        "Log-anomaly summary for incident window (2022-03-21 23:00–23:30, UTC+8):",
        f"- Earliest detected anomalous log timestamp (sample): {earliest_dt}.",
        f"- Among the examined top-20 metric-anomalous components, {num_anomalous_components} components had anomalous logs in the window.",
        f"- Components with anomalous logs (count={num_anomalous_components}): {anomalous_components_list}.",
        "",
        "- Top example components by anomalous_log_count:",
    ] + [f"  * {e}" for e in examples] + [
        "",
        "Notes:",
        "- anomalous_log_count = number of log rows in the incident window flagged high or low vs global thresholds.",
        "- distinct_log_names_with_anom = number of distinct log_name types that were anomalous.",
        "- max_log_severity = largest relative deviation from the global p95/p5 threshold observed for that component's anomalous logs.",
        "- Several components (e.g., the service-level 'adservice') showed many metric anomalies but zero anomalous logs in this window,",
        "  indicating metric issues not reflected in logs or that logs did not cross global thresholds in this time slice."
    ]

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                     cmdb_id  anomalous_log_count  distinct_log_names_with_anom  earliest_anom_timestamp  max_log_severity
0    recommendationservice-2                   40                             2             1.647875e+09          0.974359
1                adservice-0                   39                             2             1.647875e+09          0.000000
2                adservice-1                   39                             2             1.647875e+09          0.083333
3          shippingservice-2                   35                             2             1.647875e+09          0.550562
4           paymentservice-2                   35                             2             1.647875e+09          0.000000
5          currencyservice-2                   35                             2             1.647875e+09          0.930736
6          shippingservice-1                   35                             2             1.647875e+09          0.733333
7                 frontend-2                   35                             2             1.647875e+09          0.981395
8    recommendationservice-1                   34                             2             1.647875e+09          0.011407
9          shippingservice-0                   34                             2             1.647875e+09          0.310345
10         checkoutservice-0                   26                             2             1.647875e+09          0.928571
11            emailservice-1                   22                             2             1.647875e+09          0.290323
12            emailservice-2                   22                             2             1.647875e+09          0.459459
13            emailservice-0                   19                             2             1.647875e+09          0.363636
14                 adservice                    0                             0                      NaN          0.000000
15        shippingservice2-0                    0                             0                      NaN          0.000000
16  productcatalogservice2-0                    0                             0                      NaN          0.000000
17  recommendationservice2-0                    0                             0                      NaN          0.000000
18              adservice2-0                    0                             0                      NaN          0.000000
19               frontend2-0                    0                             0                      NaN          0.000000, ['recommendationservice-2', 'adservice-0', 'adservice-1', 'shippingservice-2', 'paymentservice-2', 'currencyservice-2', 'shippingservice-1', 'frontend-2', 'recommendationservice-1', 'shippingservice-0', 'checkoutservice-0', 'emailservice-1', 'emailservice-2', 'emailservice-0'], 14)```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import datetime

# Reuse df_log if present; otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure numeric values
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Incident window in UTC+8 (Asia/Shanghai)
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2022, 3, 21, 23, 0, 0))
end_dt = tz.localize(datetime.datetime(2022, 3, 21, 23, 30, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Components to restrict to
components = [
    'recommendationservice-2','adservice-0','adservice-1','shippingservice-2',
    'paymentservice-2','currencyservice-2','shippingservice-1','frontend-2',
    'recommendationservice-1','shippingservice-0','checkoutservice-0',
    'emailservice-1','emailservice-2','emailservice-0'
]

# Filter logs to incident window and selected components
df_log_window_filtered = df_log[
    (df_log['timestamp'] >= start_ts) &
    (df_log['timestamp'] <= end_ts) &
    (df_log['cmdb_id'].isin(components))
].copy()

# Aggregate per (cmdb_id, log_name)
df_log_agg = df_log_window_filtered.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    row_count = ('value', 'count'),
    mean = ('value', 'mean'),
    max = ('value', 'max'),
    earliest_timestamp = ('timestamp', 'min')
)

# Compact formatting
df_log_agg[['mean','max']] = df_log_agg[['mean','max']].round(6)

# Sort by row_count desc and keep up to 20 rows for display
df_log_agg_top20 = df_log_agg.sort_values(by='row_count', ascending=False).head(20).reset_index(drop=True)

# Collect up to 3 example rows per cmdb_id, but limit total examples to 20
examples = []
max_total_examples = 20
for cid in components:
    if len(examples) >= max_total_examples:
        break
    rows = df_log_window_filtered[df_log_window_filtered['cmdb_id'] == cid][
        ['timestamp','cmdb_id','log_name','value']
    ].sort_values(by='timestamp').head(3)
    for _, r in rows.iterrows():
        if len(examples) >= max_total_examples:
            break
        examples.append(r)

# Build examples DataFrame (may be empty)
if examples:
    df_log_examples = pd.DataFrame(examples).reset_index(drop=True)
else:
    df_log_examples = pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])

# Distinct log_name values found for these components in the window
distinct_log_names = sorted(df_log_window_filtered['log_name'].dropna().unique().tolist())
distinct_log_names_df = pd.DataFrame({'log_name': distinct_log_names})

# Final compact outputs (stored in kernel variables for further steps)
df_log_window_filtered, df_log_agg_top20, df_log_examples, distinct_log_names_df

```
Out[9]:
```
```python
# Build a concise plain-English summary from the prepared log-window and aggregates.
import datetime, pytz

try:
    df_log_window_filtered, df_log_agg_top20, df_log_examples, distinct_log_names_df
except NameError:
    summary = "Required variables not found in kernel. Re-run the log filtering/aggregation step."
else:
    tz = pytz.timezone('Asia/Shanghai')  # UTC+8

    total_rows = int(df_log_window_filtered.shape[0])
    distinct_components = int(df_log_window_filtered['cmdb_id'].nunique())
    distinct_log_names = distinct_log_names_df['log_name'].tolist()

    # Earliest timestamp across aggregates (if present)
    if not df_log_agg_top20['earliest_timestamp'].dropna().empty:
        earliest_ts = int(df_log_agg_top20['earliest_timestamp'].min())
        earliest_dt = datetime.datetime.fromtimestamp(earliest_ts, tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        earliest_dt = 'N/A'

    # Summarize a few notable aggregate rows (top 5)
    top_examples = []
    for _, r in df_log_agg_top20.head(5).iterrows():
        top_examples.append(
            f"{r['cmdb_id']} | {r['log_name']} -> row_count={int(r['row_count'])}, mean={float(r['mean']):.6f}, max={float(r['max']):.6f}"
        )

    # Example rows count
    example_count = int(df_log_examples.shape[0])

    summary_lines = [
        "Log window summary (2022-03-21 23:00–23:30, UTC+8):",
        f"- Total log rows in window (restricted components): {total_rows}.",
        f"- Distinct components present in the window: {distinct_components}.",
        f"- Distinct log_name values found: {distinct_log_names}.",
        f"- Earliest aggregate timestamp (sample): {earliest_dt}.",
        "",
        "- Top aggregate examples (cmdb_id | log_name -> row_count, mean, max):"
    ] + [f"  * {e}" for e in top_examples] + [
        "",
        f"- Example log rows returned: {example_count} rows (up to 3 per component, capped at 20 total).",
        "",
        "Notes:",
        "- Most components have one entry per minute across the 31-minute inclusive window (row_count ≈ 30–31).",
        "- Some components (e.g., frontend-2) show high log.row_count means (frontend-2 mean log.row_count ≈ 393),",
        "  while others (checkoutservice-0) have fewer rows in-window (row_count = 21) with a slightly later earliest timestamp.",
        "- Use the df_log_agg_top20 table for compact per-(cmdb_id,log_name) aggregates and df_log_examples for representative rows."
    ]

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp                  cmdb_id         log_name  value
0     1647874800              adservice-0  log.error_count    0.0
1     1647874800              adservice-0    log.row_count   36.0
2     1647874800              adservice-1  log.error_count    0.0
3     1647874800              adservice-1    log.row_count   24.0
18    1647874800        currencyservice-2  log.error_count    0.0
...          ...                      ...              ...    ...
1563  1647876600               frontend-2    log.row_count    6.0
1566  1647876600  recommendationservice-2  log.error_count    0.0
1567  1647876600  recommendationservice-2    log.row_count    1.0
1568  1647876600        shippingservice-1  log.error_count    0.0
1569  1647876600        shippingservice-1    log.row_count    2.0

[736 rows x 4 columns],                     cmdb_id         log_name  row_count        mean    max  earliest_timestamp
0         currencyservice-2  log.error_count         31    0.000000    0.0          1647874800
1         currencyservice-2    log.row_count         31  139.225806  169.0          1647874800
2                frontend-2  log.error_count         31    0.000000    0.0          1647874800
3                frontend-2    log.row_count         31  393.483871  487.0          1647874800
4   recommendationservice-2  log.error_count         31    0.000000    0.0          1647874800
5   recommendationservice-2    log.row_count         31   45.032258   53.0          1647874800
6         shippingservice-1    log.row_count         31   18.677419   32.0          1647874800
7         shippingservice-1  log.error_count         31    0.000000    0.0          1647874800
8               adservice-1    log.row_count         30   20.600000   26.0          1647874800
9               adservice-1  log.error_count         30    0.000000    0.0          1647874800
10              adservice-0  log.error_count         30    0.000000    0.0          1647874800
11              adservice-0    log.row_count         30   32.666667   36.0          1647874800
12        shippingservice-0    log.row_count         30   19.233333   36.0          1647874800
13        shippingservice-0  log.error_count         30    0.000000    0.0          1647874800
14        shippingservice-2  log.error_count         30    0.000000    0.0          1647874800
15        shippingservice-2    log.row_count         30   18.966667   32.0          1647874800
16  recommendationservice-1  log.error_count         30    0.000000    0.0          1647874800
17  recommendationservice-1    log.row_count         30   46.400000   51.0          1647874800
18        checkoutservice-0  log.error_count         21    0.000000    0.0          1647874860
19        checkoutservice-0    log.row_count         21   16.809524   34.0          1647874860,      timestamp                  cmdb_id         log_name  value
0   1647874800  recommendationservice-2  log.error_count    0.0
1   1647874800  recommendationservice-2    log.row_count   51.0
2   1647874860  recommendationservice-2  log.error_count    0.0
3   1647874800              adservice-0  log.error_count    0.0
4   1647874800              adservice-0    log.row_count   36.0
5   1647874860              adservice-0  log.error_count    0.0
6   1647874800              adservice-1  log.error_count    0.0
7   1647874800              adservice-1    log.row_count   24.0
8   1647874860              adservice-1  log.error_count    0.0
9   1647874800        shippingservice-2  log.error_count    0.0
10  1647874800        shippingservice-2    log.row_count   18.0
11  1647874860        shippingservice-2  log.error_count    0.0
12  1647874860         paymentservice-2  log.error_count    0.0
13  1647874860         paymentservice-2    log.row_count    3.0
14  1647874920         paymentservice-2  log.error_count    0.0
15  1647874800        currencyservice-2  log.error_count    0.0
16  1647874800        currencyservice-2    log.row_count  126.0
17  1647874860        currencyservice-2  log.error_count    0.0
18  1647874800        shippingservice-1  log.error_count    0.0
19  1647874800        shippingservice-1    log.row_count   12.0,           log_name
0  log.error_count
1    log.row_count)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz
import datetime
import numpy as np
import re

# Reuse existing kernel variables when available
try:
    df_metric_kpi_stats
except NameError:
    # compute global stats if missing
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
    grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    df_metric_kpi_stats = grp.agg(
        count='count',
        mean='mean',
        min='min',
        max='max',
        p95=lambda x: x.quantile(0.95),
        p5=lambda x: x.quantile(0.05)
    ).reset_index()

# Ensure df_metric is available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Candidate components list
candidates = [
    'recommendationservice-1','recommendationservice-2','frontend-2',
    'shippingservice-0','shippingservice-1','shippingservice-2',
    'adservice','adservice-0','adservice-1',
    'emailservice-0','cartservice-1','checkoutservice-0',
    'currencyservice-2','paymentservice-2'
]

# Resource-related substrings (case-insensitive)
substrs = ['cpu','memory','container','disk','io','read','write','net','packet',
           'retrans','loss','latency','fail','process','thread','gc','heap']
# build regex pattern respecting word parts
pattern = '|'.join(map(re.escape, substrs))

# Filter global metric stats to candidate components and resource-related KPIs
mask_comp = df_metric_kpi_stats['cmdb_id'].isin(candidates)
mask_kpi = df_metric_kpi_stats['kpi_name'].str.contains(pattern, case=False, na=False, regex=True)
df_resource_matches = df_metric_kpi_stats[mask_comp & mask_kpi].copy()

# Keep compact columns from global stats: cmdb_id, kpi_name, count, p95, p5
df_resource_matches_compact = df_resource_matches[['cmdb_id','kpi_name','count','p95','p5']].sort_values(
    by=['cmdb_id','kpi_name']
).reset_index(drop=True)

# Incident window UTC+8 (Asia/Shanghai): 2022-03-21 23:00 to 23:30 inclusive
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime.datetime(2022,3,21,23,0,0))
end_dt = tz.localize(datetime.datetime(2022,3,21,23,30,0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Filter full metric series to the incident window (no threshold calculation here)
df_metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Restrict to only the matched (cmdb_id, kpi_name) pairs
# Create a key set for faster filtering
matched_keys = set(zip(df_resource_matches_compact['cmdb_id'], df_resource_matches_compact['kpi_name']))
# Filter window rows by membership in matched keys
df_metric_window['key'] = list(zip(df_metric_window['cmdb_id'], df_metric_window['kpi_name']))
df_metric_window_matched = df_metric_window[df_metric_window['key'].isin(matched_keys)].copy()

# Aggregate per (cmdb_id, kpi_name) in the window
if not df_metric_window_matched.empty:
    agg_window = df_metric_window_matched.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        window_row_count = ('value','count'),
        window_mean = ('value','mean'),
        window_max = ('value','max'),
        window_min = ('value','min'),
        earliest_timestamp = ('timestamp','min')
    )
else:
    agg_window = pd.DataFrame(columns=['cmdb_id','kpi_name','window_row_count','window_mean','window_max','window_min','earliest_timestamp'])

# Join global p95/p5 onto the window aggregates (ensure global thresholds used)
df_resource_candidate_metrics = agg_window.merge(
    df_resource_matches_compact[['cmdb_id','kpi_name','p95','p5','count']].rename(columns={'count':'global_count'}),
    on=['cmdb_id','kpi_name'],
    how='left'
)

# Compute severity per group:
# sev_high = (window_max - p95)/p95 if window_max > p95 and p95!=0 else 0
# sev_low  = (p5 - window_min)/(p5 if p5!=0 else 1) if window_min < p5 else 0
def compute_severity(row):
    p95 = row['p95']
    p5 = row['p5']
    wmax = row['window_max']
    wmin = row['window_min']
    sev_h = 0.0
    sev_l = 0.0
    if pd.notna(p95) and pd.notna(wmax):
        denom = p95 if p95 != 0 else 1.0
        if wmax > p95:
            sev_h = (wmax - p95) / denom
    if pd.notna(p5) and pd.notna(wmin):
        denom5 = p5 if p5 != 0 else 1.0
        if wmin < p5:
            sev_l = (p5 - wmin) / denom5
    return max(sev_h, sev_l)

if not df_resource_candidate_metrics.empty:
    df_resource_candidate_metrics['severity'] = df_resource_candidate_metrics.apply(compute_severity, axis=1).round(6)
    # also round window_mean/max/min for compactness
    df_resource_candidate_metrics[['window_mean','window_max','window_min']] = df_resource_candidate_metrics[['window_mean','window_max','window_min']].round(6)
else:
    df_resource_candidate_metrics['severity'] = []

# Sort by severity desc and keep limit 20 for compact display
df_resource_candidate_metrics = df_resource_candidate_metrics.sort_values(by='severity', ascending=False).reset_index(drop=True)
df_resource_candidate_top20 = df_resource_candidate_metrics.head(20).copy()

# Subset where severity > 0.5 (strong deviations)
df_resource_candidate_strong = df_resource_candidate_metrics[df_resource_candidate_metrics['severity'] > 0.5].copy().reset_index(drop=True)

# Keep full results in kernel variable df_resource_candidate_metrics (already stored)
# Return compact tables: matched global stats (limit 20), the resource candidate metrics top20, and strong subset (limit 20)
df_resource_matches_compact_head = df_resource_matches_compact.head(20)

# Final outputs (compact)
df_resource_matches_compact_head, df_resource_candidate_top20, df_resource_candidate_strong.head(20)

```
Out[10]:
```
```python
# Plain-English summary built from df_resource_matches_compact_head, df_resource_candidate_top20, df_resource_candidate_strong
try:
    df_resource_candidate_top20
    df_resource_candidate_strong
    df_resource_matches_compact_head
except NameError:
    summary = "Required result tables not found in kernel. Re-run the resource candidate extraction cell."
else:
    # Top resource metric anomalies (incident window) — summarize highest severities
    top_examples = []
    for _, r in df_resource_candidate_top20.head(8).iterrows():
        top_examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} -> severity={float(r['severity']):.3f}, "
            f"window_max={r['window_max']}, p95={r['p95']}"
        )

    strong_count = int(df_resource_candidate_strong.shape[0])
    # Compose summary
    summary_lines = [
        "Resource-metric findings (candidates) — summary:",
        "",
        "1) What was selected:",
        f" - Resource-related KPIs (filtered by keywords) were matched for the requested components (sample head shown).",
        "",
        "2) Top anomalous resource metrics in the incident window (sorted by severity):",
    ] + [f" - {t}" for t in top_examples] + [
        "",
        f" - Number of candidate metric rows with strong deviation (severity > 0.5): {strong_count}.",
        "",
        "3) Key observations:",
        " - The largest deviations are network-receive related (container_network_receive_MB...) across multiple services.",
        "   Examples with very large relative spikes: emailservice-0 (severity ~20.93), checkoutservice-0 (~11.13),",
        "   shippingservice-0 (~8.43), recommendationservice-2 and -1 (~8.39 / ~8.37), adservice-1 (~7.89).",
        " - frontend-2 shows a notable network receive spike as well (severity ~3.22).",
        " - Some CPU/IO related metrics also appear with moderate severity (e.g., cartservice-1 container_cpu_cfs_throttled severity ~2.0).",
        "",
        "4) Interpretation guidance:",
        " - High severity here means the incident-window peak (or trough) deviated strongly from the group's global p95/p5.",
        " - Network receive spikes across many components suggest an unusual increase in inbound traffic or bursty transfers during the window.",
        " - Investigate the top components listed above for network I/O bottlenecks, NIC errors, or upstream traffic surges.",
        " - Also check CPU throttling (cfs throttled), memory failure counters, and filesystem read/write KPIs where present.",
        "",
        "5) Next steps (recommended):",
        " - Inspect host/container network interfaces, packet/error counters, and interface-level metrics for the noted nodes.",
        " - Correlate with traces (request spikes) and logs for the same components/time to see if traffic pattern explains the I/O spikes.",
        "",
        "Notes: The table of matched global KPIs (sample head) and the per-(cmdb_id,kpi_name) window aggregates were stored in kernel variables",
        "for further drill-down (df_resource_matches_compact_head, df_resource_candidate_top20, df_resource_candidate_strong)."
    ]

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                                           kpi_name  count           p95            p5
0   adservice  runtime.java_lang_Compilation_CompilationTimeM...     61  1.000000e+00  1.000000e+00
1   adservice  runtime.java_lang_Compilation_TotalCompilation...     61  4.703200e+04  4.703200e+04
2   adservice  runtime.java_lang_GarbageCollector_CollectionC...     61  4.095550e+04  4.082208e+04
3   adservice  runtime.java_lang_GarbageCollector_CollectionC...     61  9.000000e+00  9.000000e+00
4   adservice  runtime.java_lang_GarbageCollector_CollectionT...     61  7.444100e+04  7.420242e+04
5   adservice  runtime.java_lang_GarbageCollector_CollectionT...     61  5.730000e+02  5.730000e+02
6   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  1.000000e+00  1.000000e+00
7   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  1.000000e+00  1.000000e+00
8   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  2.333333e+00  1.250000e+00
9   adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  9.300000e+01  9.300000e+01
10  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  7.321025e+08  7.288663e+08
11  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  7.234851e+08  7.234851e+08
12  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  4.095550e+04  4.082208e+04
13  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  9.000000e+00  9.000000e+00
14  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  2.333082e+07  2.333082e+07
15  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  2.333082e+07  2.333082e+07
16  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  3.932160e+06  3.932160e+06
17  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  3.932160e+06  3.932160e+06
18  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  2.988442e+07  2.988442e+07
19  adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...     61  2.765619e+07  2.765619e+07,                     cmdb_id                                           kpi_name  window_row_count  window_mean   window_max  window_min  earliest_timestamp          p95          p5  global_count   severity
0            emailservice-0  container.node-6.container_network_receive_MB....                31     0.043363     0.620644    0.021307          1647874800     0.028298    0.021505            61  20.932420
1         checkoutservice-0  container.node-6.container_network_receive_MB....                31     0.054410     0.612411    0.020482          1647874800     0.050478    0.024778            61  11.132236
2         shippingservice-0  container.node-6.container_network_receive_MB....                31     0.046784     0.322267    0.020192          1647874800     0.034172    0.022308            61   8.430720
3   recommendationservice-2  container.node-6.container_network_receive_MB....                31     0.074481     0.627833    0.045982          1647874800     0.066886    0.048079            61   8.386619
4   recommendationservice-1  container.node-6.container_network_receive_MB....                31     0.074755     0.638567    0.043936          1647874800     0.068154    0.046437            61   8.369479
5               adservice-1  container.node-6.container_network_receive_MB....                31     0.049625     0.329134    0.024025          1647874800     0.037015    0.026598            61   7.891921
6                frontend-2  container.node-5.container_network_receive_MB....                31     0.135480     0.681479    0.076634          1647874800     0.161362    0.087079            61   3.223290
7             cartservice-1  container.node-6.container_cpu_cfs_throttled_p...                31     0.064516     2.000000    0.000000          1647874800     0.000000    0.000000            61   2.000000
8         shippingservice-2  container.node-5.container_network_receive_MB....                31     0.055117     0.594010    0.017622          1647874800     0.225045    0.018649            61   1.639518
9         checkoutservice-0       container.node-6.container_fs_reads./dev/vda                31     0.096774     1.500000    0.000000          1647874800     0.000000    0.000000            61   1.500000
10              adservice-0  mesh.destination.frontend.adservice.istio_requ...                31     0.028935     0.078250    0.019400          1647874800     0.035625    0.020400            61   1.196491
11              adservice-1  mesh.destination.frontend.adservice.istio_requ...                31     0.032639     0.127425    0.020925          1647874800     0.060300    0.020950            61   1.113184
12         paymentservice-2        container.node-6.container_cpu_user_seconds                31     0.000968     0.010000    0.000000          1647874800     0.005000    0.000000            61   1.000000
13               frontend-2  mesh.source.frontend.jaeger-collector.istio_re...                31     0.002156     0.009275    0.000000          1647874800     0.005625    0.000525            61   1.000000
14         paymentservice-2  container.node-6.container_memory_failures.hie...                31     6.919355    66.000000    0.000000          1647874800    33.000000    0.000000            61   1.000000
15        shippingservice-1  mesh.source.shippingservice.jaeger-collector.i...                31     0.000068     0.001050    0.000000          1647874800     0.000525    0.000000            61   1.000000
16  recommendationservice-1      container.node-6.container_cpu_system_seconds                31     0.011452     0.020000    0.000000          1647874800     0.020000    0.005000            61   1.000000
17        shippingservice-2  mesh.destination.checkoutservice.shippingservi...                31   981.935484  2175.000000    0.000000          1647874800  1812.500000  362.500000            61   1.000000
18        shippingservice-1  mesh.destination.checkoutservice.shippingservi...                31  1027.419355  2537.500000    0.000000          1647874800  1812.500000  357.500000            61   1.000000
19        shippingservice-2  mesh.destination.checkoutservice.shippingservi...                31  1693.548387  3750.000000    0.000000          1647874800  3125.000000  625.000000            61   1.000000,                     cmdb_id                                           kpi_name  window_row_count  window_mean   window_max  window_min  earliest_timestamp          p95          p5  global_count   severity
0            emailservice-0  container.node-6.container_network_receive_MB....                31     0.043363     0.620644    0.021307          1647874800     0.028298    0.021505            61  20.932420
1         checkoutservice-0  container.node-6.container_network_receive_MB....                31     0.054410     0.612411    0.020482          1647874800     0.050478    0.024778            61  11.132236
2         shippingservice-0  container.node-6.container_network_receive_MB....                31     0.046784     0.322267    0.020192          1647874800     0.034172    0.022308            61   8.430720
3   recommendationservice-2  container.node-6.container_network_receive_MB....                31     0.074481     0.627833    0.045982          1647874800     0.066886    0.048079            61   8.386619
4   recommendationservice-1  container.node-6.container_network_receive_MB....                31     0.074755     0.638567    0.043936          1647874800     0.068154    0.046437            61   8.369479
5               adservice-1  container.node-6.container_network_receive_MB....                31     0.049625     0.329134    0.024025          1647874800     0.037015    0.026598            61   7.891921
6                frontend-2  container.node-5.container_network_receive_MB....                31     0.135480     0.681479    0.076634          1647874800     0.161362    0.087079            61   3.223290
7             cartservice-1  container.node-6.container_cpu_cfs_throttled_p...                31     0.064516     2.000000    0.000000          1647874800     0.000000    0.000000            61   2.000000
8         shippingservice-2  container.node-5.container_network_receive_MB....                31     0.055117     0.594010    0.017622          1647874800     0.225045    0.018649            61   1.639518
9         checkoutservice-0       container.node-6.container_fs_reads./dev/vda                31     0.096774     1.500000    0.000000          1647874800     0.000000    0.000000            61   1.500000
10              adservice-0  mesh.destination.frontend.adservice.istio_requ...                31     0.028935     0.078250    0.019400          1647874800     0.035625    0.020400            61   1.196491
11              adservice-1  mesh.destination.frontend.adservice.istio_requ...                31     0.032639     0.127425    0.020925          1647874800     0.060300    0.020950            61   1.113184
12         paymentservice-2        container.node-6.container_cpu_user_seconds                31     0.000968     0.010000    0.000000          1647874800     0.005000    0.000000            61   1.000000
13               frontend-2  mesh.source.frontend.jaeger-collector.istio_re...                31     0.002156     0.009275    0.000000          1647874800     0.005625    0.000525            61   1.000000
14         paymentservice-2  container.node-6.container_memory_failures.hie...                31     6.919355    66.000000    0.000000          1647874800    33.000000    0.000000            61   1.000000
15        shippingservice-1  mesh.source.shippingservice.jaeger-collector.i...                31     0.000068     0.001050    0.000000          1647874800     0.000525    0.000000            61   1.000000
16  recommendationservice-1      container.node-6.container_cpu_system_seconds                31     0.011452     0.020000    0.000000          1647874800     0.020000    0.005000            61   1.000000
17        shippingservice-2  mesh.destination.checkoutservice.shippingservi...                31   981.935484  2175.000000    0.000000          1647874800  1812.500000  362.500000            61   1.000000
18        shippingservice-1  mesh.destination.checkoutservice.shippingservi...                31  1027.419355  2537.500000    0.000000          1647874800  1812.500000  357.500000            61   1.000000
19        shippingservice-2  mesh.destination.checkoutservice.shippingservi...                31  1693.548387  3750.000000    0.000000          1647874800  3125.000000  625.000000            61   1.000000)```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# Reuse existing kernel variables where available
# Candidate components list
candidates = [
    'recommendationservice-1','recommendationservice-2','frontend-2',
    'shippingservice-0','shippingservice-1','shippingservice-2',
    'adservice','adservice-0','adservice-1',
    'emailservice-0','cartservice-1','checkoutservice-0',
    'currencyservice-2','paymentservice-2'
]

# Source: df_cmdb_agg (metric-level aggregation) or fallback to df_cmdb_agg_top
if 'df_cmdb_agg' in globals():
    df_cmdb_src = df_cmdb_agg
elif 'df_cmdb_agg_top' in globals():
    df_cmdb_src = df_cmdb_agg_top
else:
    # create empty with expected cols to avoid merge errors
    df_cmdb_src = pd.DataFrame(columns=['cmdb_id','distinct_kpi_anom_count','total_anomaly_count'])

# Ensure df_component_trace_summary exists
if 'df_component_trace_summary' not in globals():
    df_component_trace_summary = pd.DataFrame(columns=[
        'cmdb_id','distinct_kpi_anom_count','total_anomaly_count',
        'trace_to_anom_count','trace_from_anom_count','trace_self_anom_count',
        'trace_earliest_anom_timestamp','max_trace_severity'
    ])

# Ensure df_resource_candidate_top20 exists
if 'df_resource_candidate_top20' not in globals():
    df_resource_candidate_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','window_row_count','window_mean','window_max','window_min',
        'earliest_timestamp','p95','p5','global_count','severity'
    ])

# Ensure df_log_anoms_by_component exists
if 'df_log_anoms_by_component' not in globals():
    df_log_anoms_by_component = pd.DataFrame(columns=[
        'cmdb_id','anomalous_log_count','distinct_log_names_with_anom','earliest_anom_timestamp','max_log_severity'
    ])

# Prepare metric-level columns for candidates
df_metrics_part = df_cmdb_src[['cmdb_id','distinct_kpi_anom_count','total_anomaly_count']].copy()
# Filter to candidates and ensure one row per cmdb_id (if duplicates, take first)
df_metrics_part = df_metrics_part[df_metrics_part['cmdb_id'].isin(candidates)].drop_duplicates(subset=['cmdb_id']).set_index('cmdb_id')

# Prepare trace part (ensure columns present)
trace_cols = ['cmdb_id','trace_to_anom_count','trace_from_anom_count','trace_self_anom_count','max_trace_severity','trace_earliest_anom_timestamp']
df_trace_part = df_component_trace_summary.copy()
for c in trace_cols:
    if c not in df_trace_part.columns and c != 'cmdb_id':
        df_trace_part[c] = 0 if 'count' in c or 'severity' in c else pd.NA
df_trace_part = df_trace_part[['cmdb_id','trace_to_anom_count','trace_from_anom_count','trace_self_anom_count','max_trace_severity','trace_earliest_anom_timestamp']].copy()
df_trace_part = df_trace_part[df_trace_part['cmdb_id'].isin(candidates)].set_index('cmdb_id')

# Prepare resource network entries: pick kpi_name containing 'container_network_receive' (case-insensitive)
mask_net = df_resource_candidate_top20['kpi_name'].str.contains('container_network_receive', case=False, na=False)
df_resource_net = df_resource_candidate_top20[mask_net].copy()
# For each cmdb_id pick the row with max severity (if multiple)
if not df_resource_net.empty:
    df_resource_net = df_resource_net.sort_values(['cmdb_id','severity'], ascending=[True, False])
    df_resource_net = df_resource_net.groupby('cmdb_id', as_index=False).first()[['cmdb_id','kpi_name','severity','earliest_timestamp']]
    df_resource_net = df_resource_net.set_index('cmdb_id')
else:
    df_resource_net = pd.DataFrame(columns=['kpi_name','severity','earliest_timestamp'])
    df_resource_net.index.name = 'cmdb_id'

# Prepare log anomaly part
log_cols = ['cmdb_id','anomalous_log_count','distinct_log_names_with_anom','max_log_severity','earliest_anom_timestamp']
df_log_part = df_log_anoms_by_component.copy()
# rename to match requested output names where necessary
rename_map = {
    'max_log_severity':'log_max_severity',
    'anomalous_log_count':'anomalous_log_count',
    'distinct_log_names_with_anom':'log_distinct_names_with_anom',
    'earliest_anom_timestamp':'log_earliest_anom_timestamp'
}
for old,new in rename_map.items():
    if old in df_log_part.columns:
        df_log_part = df_log_part.rename(columns={old:new})
# Ensure columns exist
expected_log_cols = ['cmdb_id','anomalous_log_count','log_distinct_names_with_anom','log_max_severity','log_earliest_anom_timestamp']
for c in expected_log_cols:
    if c not in df_log_part.columns:
        df_log_part[c] = 0 if 'count' in c or 'max_severity' in c else pd.NA
df_log_part = df_log_part[expected_log_cols].copy().set_index('cmdb_id')

# Build a base DataFrame of candidates to preserve order
df_base = pd.DataFrame({'cmdb_id': candidates}).set_index('cmdb_id')

# Merge all parts onto base
df_candidate_summary = df_base.join(df_metrics_part[['distinct_kpi_anom_count','total_anomaly_count']], how='left')
df_candidate_summary = df_candidate_summary.join(df_trace_part, how='left')
# resource fields: resource_kpi_name, resource_network_severity, resource_earliest_timestamp
df_candidate_summary = df_candidate_summary.join(df_resource_net[['kpi_name','severity','earliest_timestamp']].rename(
    columns={'kpi_name':'resource_kpi_name','severity':'resource_network_severity','earliest_timestamp':'resource_earliest_timestamp'}
), how='left')
df_candidate_summary = df_candidate_summary.join(df_log_part.rename(columns={
    'anomalous_log_count':'anomalous_log_count',
    'log_distinct_names_with_anom':'log_distinct_names_with_anom',
    'log_max_severity':'log_max_severity',
    'log_earliest_anom_timestamp':'log_earliest_anom_timestamp'
}), how='left')

# Fill missing numeric fields with 0 and timestamps with pd.NA
num_cols = ['distinct_kpi_anom_count','total_anomaly_count','trace_to_anom_count','trace_from_anom_count','trace_self_anom_count','max_trace_severity','resource_network_severity','anomalous_log_count','log_distinct_names_with_anom','log_max_severity']
for c in num_cols:
    if c in df_candidate_summary.columns:
        df_candidate_summary[c] = df_candidate_summary[c].fillna(0)
# Timestamps
ts_cols = ['trace_earliest_anom_timestamp','resource_earliest_timestamp','log_earliest_anom_timestamp']
for c in ts_cols:
    if c in df_candidate_summary.columns:
        df_candidate_summary[c] = df_candidate_summary[c].where(df_candidate_summary[c].notna(), pd.NA)

# Reset index to show cmdb_id as column
df_candidate_summary = df_candidate_summary.reset_index()

# Reorder columns as requested
cols_order = [
    'cmdb_id','distinct_kpi_anom_count','total_anomaly_count',
    'trace_to_anom_count','trace_from_anom_count','trace_self_anom_count','max_trace_severity',
    'resource_kpi_name','resource_network_severity','resource_earliest_timestamp',
    'anomalous_log_count','log_distinct_names_with_anom','log_max_severity'
]
# Ensure presence
present_cols = [c for c in cols_order if c in df_candidate_summary.columns]
df_candidate_summary = df_candidate_summary[present_cols]

# Sort by resource_network_severity desc then trace_to_anom_count desc
df_candidate_summary = df_candidate_summary.sort_values(by=['resource_network_severity','trace_to_anom_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Count how many components have resource_network_severity > 1 and trace_to_anom_count > 0
count_high_resource_and_trace_to = int(((df_candidate_summary['resource_network_severity'] > 1) & (df_candidate_summary.get('trace_to_anom_count', 0) > 0)).sum())

# Store result in kernel variable df_candidate_summary (already assigned)
df_candidate_summary, count_high_resource_and_trace_to

```
Out[11]:
```
```python
# Build a plain-English summary from df_candidate_summary and count_high_resource_and_trace_to
import pandas as pd

# Use existing kernel variables
try:
    df_candidate_summary
    count_high_resource_and_trace_to
except NameError:
    summary = "Required summary table not found in kernel."
else:
    # Identify components with resource_network_severity > 1 and trace_to_anom_count > 0
    mask = (df_candidate_summary['resource_network_severity'] > 1) & (df_candidate_summary['trace_to_anom_count'] > 0)
    downstream_candidates = df_candidate_summary.loc[mask, ['cmdb_id','resource_network_severity','trace_to_anom_count','anomalous_log_count']]
    top_resource = df_candidate_summary.sort_values('resource_network_severity', ascending=False).head(6)[
        ['cmdb_id','resource_network_severity','trace_to_anom_count','anomalous_log_count']
    ]

    lines = []
    lines.append("Cross-source candidate summary (concise):")
    lines.append(f"- Components examined: {len(df_candidate_summary)} (selected candidate list).")
    lines.append(f"- Components with strong resource-network deviation (>1) AND upstream trace anomalies (trace_to_anom_count>0): {int(count_high_resource_and_trace_to)}.")
    if not downstream_candidates.empty:
        lines.append("  -> These components are:")
        for _, r in downstream_candidates.iterrows():
            lines.append(f"     • {r['cmdb_id']}: resource_network_severity={r['resource_network_severity']:.3f}, trace_to_anom_count={int(r['trace_to_anom_count'])}, anomalous_log_count={int(r['anomalous_log_count'])}")
    else:
        lines.append("  -> None matched the combined criterion.")

    lines.append("")
    lines.append("Top components by resource network severity (sample):")
    for _, r in top_resource.iterrows():
        lines.append(f"  • {r['cmdb_id']}: net_sev={r['resource_network_severity']:.3f}, trace_to={int(r['trace_to_anom_count'])}, anomalous_logs={int(r['anomalous_log_count'])}")

    lines.append("")
    lines.append("Key takeaways:")
    lines.append(" - Multiple components show large network receive spikes during the incident window (notably emailservice-0, checkoutservice-0, shippingservice-0, recommendationservice-2/1, adservice-1, frontend-2).")
    lines.append(" - The strongest actionable candidates (both high network severity and upstream trace activity) are: checkoutservice-0, recommendationservice-2, recommendationservice-1, and frontend-2.")
    lines.append(" - Some components (e.g., adservice at service level) have many metric anomalies but little trace or log evidence in this window — investigate metrics/hosts/logs separately.")
    lines.append(" - Next steps: focus on the 4 components above for correlation across traces, network I/O on their nodes, and relevant logs; then expand to other high-severity network spikes.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id  distinct_kpi_anom_count  total_anomaly_count  trace_to_anom_count  trace_from_anom_count  trace_self_anom_count  max_trace_severity                                  resource_kpi_name  resource_network_severity  resource_earliest_timestamp  anomalous_log_count  log_distinct_names_with_anom  log_max_severity
0            emailservice-0                       64                 1735                  0.0                   52.0                    0.0            0.089231  container.node-6.container_network_receive_MB....                  20.932420                 1.647875e+09                 19.0                           2.0          0.363636
1         checkoutservice-0                       64                 1622                591.0                   78.0                   27.0            0.904762  container.node-6.container_network_receive_MB....                  11.132236                 1.647875e+09                 26.0                           2.0          0.928571
2         shippingservice-0                       64                 1742                  0.0                  203.0                    0.0            0.815951  container.node-6.container_network_receive_MB....                   8.430720                 1.647875e+09                 34.0                           2.0          0.310345
3   recommendationservice-2                       64                 1721                153.0                   94.0                   42.0            0.243902  container.node-6.container_network_receive_MB....                   8.386619                 1.647875e+09                 40.0                           2.0          0.974359
4   recommendationservice-1                       64                 1686                156.0                   94.0                   48.0            5.463415  container.node-6.container_network_receive_MB....                   8.369479                 1.647875e+09                 34.0                           2.0          0.011407
5               adservice-1                       64                 1641                  0.0                  107.0                    0.0            1.571429  container.node-6.container_network_receive_MB....                   7.891921                 1.647875e+09                 39.0                           2.0          0.083333
6                frontend-2                       64                 1722               1010.0                   43.0                   42.0            2.406593  container.node-5.container_network_receive_MB....                   3.223290                 1.647875e+09                 35.0                           2.0          0.981395
7         shippingservice-2                       64                 1747                  0.0                  186.0                    0.0            2.406593  container.node-5.container_network_receive_MB....                   1.639518                 1.647875e+09                 35.0                           2.0          0.550562
8         shippingservice-1                       64                 1739                  0.0                  184.0                    0.0            1.450549                                                NaN                   0.000000                          NaN                 35.0                           2.0          0.733333
9                 adservice                      340                 9700                  0.0                    0.0                    0.0            0.000000                                                NaN                   0.000000                          NaN                  0.0                           0.0          0.000000
10              adservice-0                       64                 1665                  0.0                  115.0                    0.0            1.423077                                                NaN                   0.000000                          NaN                 39.0                           2.0          0.000000
11            cartservice-1                       64                 1572                  0.0                    0.0                    0.0            0.000000                                                NaN                   0.000000                          NaN                  0.0                           0.0          0.000000
12        currencyservice-2                       64                 1600                  0.0                  215.0                    0.0            2.073892                                                NaN                   0.000000                          NaN                 35.0                           2.0          0.930736
13         paymentservice-2                       64                 1655                  0.0                   56.0                    0.0            0.303502                                                NaN                   0.000000                          NaN                 35.0                           2.0          0.000000, 4)```
```

In [ ]:
# In[12]:

import pandas as pd
import re

# Reuse df_trace_anoms from kernel
try:
    df_trace_anoms
except NameError:
    raise RuntimeError("df_trace_anoms not found in kernel. Run trace anomaly detection first.")

# Candidate components list
candidates = {
    'recommendationservice-1','recommendationservice-2','frontend-2',
    'shippingservice-0','shippingservice-1','shippingservice-2',
    'adservice','adservice-0','adservice-1',
    'emailservice-0','cartservice-1','checkoutservice-0',
    'currencyservice-2','paymentservice-2'
}

# Filter rows where trace_name contains 'trace.to_'
mask_to = df_trace_anoms['trace_name'].str.contains(r'trace\.to_', na=False)
df_trace_to = df_trace_anoms[mask_to].copy()

# Parse target service from trace_name: substring after 'trace.to_' up to next '.' (or end)
def parse_target(trace_name):
    m = re.search(r'trace\.to_([^\.]+)', str(trace_name))
    return m.group(1) if m else None

df_trace_to['target_cmdb_id'] = df_trace_to['trace_name'].apply(parse_target)
df_trace_to['source_cmdb_id'] = df_trace_to['cmdb_id']

# Aggregate edges by (source, target) using anomaly_count, earliest_anom_timestamp, severity
agg = df_trace_to.groupby(['source_cmdb_id','target_cmdb_id'], as_index=False).agg(
    anomaly_edge_count = ('anomaly_count', 'sum'),
    earliest_anom_timestamp = ('earliest_anom_timestamp', 'min'),
    max_edge_severity = ('severity', 'max')
)

# Restrict to edges where either source or target is in the candidate set
agg_filtered = agg[
    (agg['source_cmdb_id'].isin(candidates)) | (agg['target_cmdb_id'].isin(candidates))
].copy()

# Store full edges in kernel variable
df_trace_edges = agg_filtered.copy()

# Prepare compact top-20 table sorted by anomaly_edge_count desc
df_trace_edges_top20 = df_trace_edges.sort_values(by='anomaly_edge_count', ascending=False).head(20).reset_index(drop=True)

# Find distinct edges where both source and target are in the candidate list
mask_both = df_trace_edges['source_cmdb_id'].isin(candidates) & df_trace_edges['target_cmdb_id'].isin(candidates)
edges_both = df_trace_edges[mask_both].copy().reset_index(drop=True)
edges_both_count = int(edges_both.shape[0])
edges_both_df = edges_both[['source_cmdb_id','target_cmdb_id','anomaly_edge_count','earliest_anom_timestamp','max_edge_severity']].head(20)

# Compact outputs for the Controller to read
df_trace_edges_top20, edges_both_df, edges_both_count

```
Out[12]:
```
Summary of trace "to" anomaly edges (incident window):

- Total top edges (sample, sorted by anomaly_edge_count):
  1. frontend-2 -> cartservice-1: 83 anomaly rows, earliest 1647874800, max severity 0.65
  2. frontend-2 -> cartservice-0: 70 anomaly rows, earliest 1647874800, max severity 0.144
  3. frontend-0 -> cartservice-1: 67 anomaly rows, earliest 1647874800, max severity 0.1667
  4. frontend-2 -> cartservice-2: 67 anomaly rows, earliest 1647874800, max severity 1.30
  5. frontend-0 -> adservice-0: 59 anomaly rows, earliest 1647874800, max severity 1.423
  6. checkoutservice-0 -> cartservice-2: 56 anomaly rows, earliest 1647874860, max severity 0.0
  7. frontend-2 -> adservice-0: 56 anomaly rows, earliest 1647874800, max severity 1.211
  8. recommendationservice-1 -> productcatalogservice-2: 56 anomaly rows, earliest 1647874800, max severity 0.154
  9. frontend-2 -> adservice-2: 55 anomaly rows, earliest 1647874800, max severity 0.893
  10. frontend-2 -> adservice-1: 54 anomaly rows, earliest 1647874800, max severity 0.575
  11. frontend-2 -> shippingservice-0: 53 anomaly rows, earliest 1647874800, max severity 0.10
  12. frontend-0 -> adservice-1: 53 anomaly rows, earliest 1647874800, max severity 1.571
  13. checkoutservice-0 -> cartservice-0: 52 anomaly rows, earliest 1647874860, max severity 0.429
  14. recommendationservice-2 -> productcatalogservice-1: 52 anomaly rows, earliest 1647874800, max severity 0.244
  15. checkoutservice-1 -> cartservice-1: 52 anomaly rows, earliest 1647874800, max severity 0.429
  16. checkoutservice-0 -> cartservice-1: 52 anomaly rows, earliest 1647874860, max severity 0.429
  17. recommendationservice-2 -> productcatalogservice-0: 51 anomaly rows, earliest 1647874800, max severity 0.130
  18. recommendationservice-2 -> productcatalogservice-2: 50 anomaly rows, earliest 1647874800, max severity 0.136
  19. recommendationservice-1 -> productcatalogservice-0: 50 anomaly rows, earliest 1647874800, max severity 5.463  (notably high)
  20. recommendationservice-1 -> productcatalogservice-1: 50 anomaly rows, earliest 1647874800, max severity 0.130

- Distinct edges where both source AND parsed target are in the provided candidate list:
  - Count: 17

  - These edges (source -> target):
    1. checkoutservice-0 -> cartservice-1
    2. checkoutservice-0 -> currencyservice-2
    3. checkoutservice-0 -> emailservice-0
    4. checkoutservice-0 -> paymentservice-2
    5. checkoutservice-0 -> shippingservice-0
    6. checkoutservice-0 -> shippingservice-1
    7. checkoutservice-0 -> shippingservice-2
    8. frontend-2 -> adservice-0
    9. frontend-2 -> adservice-1
    10. frontend-2 -> cartservice-1
    11. frontend-2 -> checkoutservice-0
    12. frontend-2 -> currencyservice-2
    13. frontend-2 -> recommendationservice-1
    14. frontend-2 -> recommendationservice-2
    15. frontend-2 -> shippingservice-0
    16. frontend-2 -> shippingservice-1
    17. frontend-2 -> shippingservice-2

Key observations and guidance:
- frontend-2 is a major source of anomalous edges to many candidate services (multiple edges, high counts), including strong severity on some edges (e.g., frontend-2 -> shippingservice-2 severity ~2.41; frontend-2 -> currencyservice-2 severity ~1.69; frontend-2 -> adservice-0 severity ~1.21).
- recommendationservice-1 -> productcatalogservice-0 shows a very high edge severity (~5.46) despite a moderate edge count (50) — worth investigation.
- Many checkoutservice-0 -> cartservice/currency/payment/shipping edges exist (checkoutservice-0 interacts widely), with several edges having substantial anomaly counts (50+).
- Earliest anomaly timestamps are concentrated at 1647874800 (start of incident window), with some checkout-related edges earliest at 1647874860.

Suggested next steps:
- Prioritize investigation of frontend-2 (high-degree source) and edges with both high count and high severity (e.g., frontend-2 -> shippingservice-2, frontend-2 -> currencyservice-2, frontend-2 -> adservice-0, recommendationservice-1 -> productcatalogservice-0).
- Correlate these edges with resource-network spikes and logs for the involved source/target components to determine upstream vs downstream root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(             source_cmdb_id           target_cmdb_id  anomaly_edge_count  earliest_anom_timestamp  max_edge_severity
0                frontend-2            cartservice-1                  83               1647874800           0.650000
1                frontend-2            cartservice-0                  70               1647874800           0.144000
2                frontend-0            cartservice-1                  67               1647874800           0.166667
3                frontend-2            cartservice-2                  67               1647874800           1.300000
4                frontend-0              adservice-0                  59               1647874800           1.423077
5         checkoutservice-0            cartservice-2                  56               1647874860           0.000000
6                frontend-2              adservice-0                  56               1647874800           1.210526
7   recommendationservice-1  productcatalogservice-2                  56               1647874800           0.153846
8                frontend-2              adservice-2                  55               1647874800           0.892857
9                frontend-2              adservice-1                  54               1647874800           0.575342
10               frontend-2        shippingservice-0                  53               1647874800           0.100000
11               frontend-0              adservice-1                  53               1647874800           1.571429
12        checkoutservice-0            cartservice-0                  52               1647874860           0.428571
13  recommendationservice-2  productcatalogservice-1                  52               1647874800           0.243902
14        checkoutservice-1            cartservice-1                  52               1647874800           0.428571
15        checkoutservice-0            cartservice-1                  52               1647874860           0.428571
16  recommendationservice-2  productcatalogservice-0                  51               1647874800           0.130435
17  recommendationservice-2  productcatalogservice-2                  50               1647874800           0.136364
18  recommendationservice-1  productcatalogservice-0                  50               1647874800           5.463415
19  recommendationservice-1  productcatalogservice-1                  50               1647874800           0.130435,        source_cmdb_id           target_cmdb_id  anomaly_edge_count  earliest_anom_timestamp  max_edge_severity
0   checkoutservice-0            cartservice-1                  52               1647874860           0.428571
1   checkoutservice-0        currencyservice-2                  44               1647874860           0.066667
2   checkoutservice-0           emailservice-0                  18               1647875100           0.089231
3   checkoutservice-0         paymentservice-2                  18               1647875040           0.291785
4   checkoutservice-0        shippingservice-0                  35               1647874860           0.428571
5   checkoutservice-0        shippingservice-1                  28               1647874860           0.178947
6   checkoutservice-0        shippingservice-2                  32               1647874860           0.863636
7          frontend-2              adservice-0                  56               1647874800           1.210526
8          frontend-2              adservice-1                  54               1647874800           0.575342
9          frontend-2            cartservice-1                  83               1647874800           0.650000
10         frontend-2        checkoutservice-0                  26               1647874860           0.113941
11         frontend-2        currencyservice-2                  43               1647874800           1.694444
12         frontend-2  recommendationservice-1                  46               1647874800           0.200000
13         frontend-2  recommendationservice-2                  49               1647874800           0.059603
14         frontend-2        shippingservice-0                  53               1647874800           0.100000
15         frontend-2        shippingservice-1                  47               1647874800           0.333333
16         frontend-2        shippingservice-2                  49               1647874800           2.406593, 17)```
```